# Hinode/EIS データ解析講習会（通し版）

Warren, Winebarger & Brooks (2012), ApJ 759, 141 と同じ解析を最後まで通します。

**この 1 冊にモジュール 0〜7 が全部入っています。** 上から順に実行してください。
インストールとデータ取得は**最初の 1 回だけ**で済みます。

| | 内容 | 目安 |
|---|---|---|
| 0 | 環境構築とデータ取得 | 5 分（EIS 94 MB のダウンロード込み） |
| 1 | EIS のデータを見る | 40 分 |
| 2 | スペクトル線フィット | 50 分 |
| 3 | AIA 94 → Fe XVIII | 50 分 |
| 4 | 座標合わせと箱の選択 | 40 分 |
| 5 | **論文 Table 2 と答え合わせ** | 30 分 |
| 6 | 寄与関数と EM loci | 40 分 |
| 7 | DEM インバージョン | 60 分 |

半日コースは 5 まで、1 日コースは 7 まで。
モジュールごとに分かれた版は
[`notebooks/`](https://github.com/hottahd/EIS_practice/tree/main/notebooks) にあります。

**★ Colab の保存について**: GitHub から開いたノートは読み取り専用の一時セッションです。
編集や実行結果を残したいときは「ファイル → ドライブにコピーを保存」。
仮想マシンが切れるとダウンロードしたデータも消えますが、
その場合は上から流し直せば復帰できます（既にあるファイルは取得し直しません）。


## 準備（この 1 冊で 1 回だけ）

パッケージを入れて、教材リポジトリを取ってくる。
**観測データは、必要になったところで各モジュールが自分で取得する**
（既にあれば何もしないので、上から流し直しても無駄が無い）。

In [ ]:
!pip install -q eispac fiasco demregpy

### インストール直後のランタイム再起動について
#
Colab では、pip が `numpy` などを入れ替えると、**実行中のセッションが
古いモジュールを掴んだまま**になり、あとで次のようなエラーが出ることがある:
#
```
ImportError: cannot import name '_center' from 'numpy._core.umath'
```
#
これはインストールの失敗ではなく、**再起動すれば直る**。
次のセルが入れ替えを検出して、必要なときだけ自動で再起動する。
#
**再起動が起きたら、もう一度このノートを先頭から実行すること。**
2 回目はインストールもダウンロードも済んでいるので一瞬で終わる。

In [ ]:
import sys
from importlib.metadata import version

need_restart = False
try:
    loaded = sys.modules["numpy"].__version__ if "numpy" in sys.modules else None
    if loaded is not None and loaded != version("numpy"):
        need_restart = True
        print(f"numpy が {loaded} -> {version('numpy')} に入れ替わりました")
except Exception as e:                      # 判定自体が失敗したら念のため再起動
    need_restart = True
    print("numpy の状態を確認できませんでした:", e)

if need_restart:
    print("ランタイムを再起動します。"
          "再起動したら、もう一度このノートを先頭から実行してください。")
    try:
        import IPython
        ipy = IPython.get_ipython()
        if ipy is not None:
            ipy.kernel.do_shutdown(True)    # Colab のランタイム再起動
    except Exception:
        import os
        os.kill(os.getpid(), 9)
else:
    print("numpy の入れ替えは起きていません。このまま先へ進んで大丈夫です。")

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/hottahd/EIS_practice.git"
if not os.path.exists("scripts/lines_warren2012.py"):      # リポジトリの外にいる
    if not os.path.exists("EIS_practice"):
        print("教材リポジトリを取得中 ...")
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("EIS_practice")
sys.path.insert(0, "scripts")
print("作業ディレクトリ:", os.getcwd())

# モジュール 0: 環境構築とデータ取得

Hinode/EIS データ解析講習会 — Warren, Winebarger & Brooks (2012) を再現する

**このノートで何をするか**

1. 必要なパッケージを入れる（すべて pip、5 分程度）
2. 観測データを取る（EIS 94 MB + AIA 3 MB、**ユーザ登録は不要**）
3. 動作確認

**題材**: 2011 年 7 月 2 日 03:07 UT、活動領域 NOAA 1243
（論文 Table 1 の region 7）。**論文に観測値の表が載っている唯一の活動領域**なので、
自分の解析結果を 1 行ずつ答え合わせできる。

## 0-1. パッケージを入れる

| パッケージ | 用途 |
|---|---|
| `eispac` | EIS の level-1 HDF5 を読む、輝線フィット |
| `sunpy` | AIA の画像を扱う |
| `fiasco` | CHIANTI の原子データから寄与関数 G(T) を作る |
| `demregpy` | 正則化インバージョンで DEM を解く |

`aiapy` は**入れない**。この教材が使う AIA は JSOC の synoptic 版で、
既に level-1.5 なので `aiapy` の前処理が要らないため
（`aiapy` は numpy≥2.1 を要求し、Colab に元から入っている `numba` と衝突する）。
フルディスクの level-1 を自分で処理したい人だけ `pip install aiapy` すればよい。

### ★ 赤い `ERROR:` が出ても、たいていは無視してよい

Colab では、上のインストールで次のような行が出ることがある:

```
ERROR: pip's dependency resolver does not currently take into account all the
packages that are installed. ...
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2
```

これは**インストールの失敗ではない**。pip が
「Colab に元から入っている別のパッケージが要求するバージョンと食い違っている」
と報告しているだけで、教材で使うパッケージ自体は正しく入っている。

- `requests` は `sunpy` が 2.33 以上を要求するため上がる。
  Colab 自身の機能（Drive 連携など）が使うものだが、この教材では問題にならない。
- `numpy` / `numba` の衝突が出る場合は `aiapy` が原因なので、**入れていない**（上記）。

**次のセルでバージョンが表示されれば問題なし。**
もし後のセルで `numpy` 関連のエラーが出たら、
**「ランタイム」→「セッションの再起動」** をして、**先頭から実行し直す**こと
（再起動してもダウンロード済みのファイルは残るので、待ち時間はほとんど無い）。

In [ ]:
import eispac, sunpy, fiasco, demregpy
import numpy as np, matplotlib.pyplot as plt
print("eispac  ", eispac.__version__)
print("sunpy   ", sunpy.__version__)
print("fiasco  ", fiasco.__version__)
print("demregpy", demregpy.__version__ if hasattr(demregpy, "__version__") else "(ok)")
print("numpy   ", np.__version__)

## 0-2. 教材リポジトリを取ってくる

スクリプトと、IDL/SolarSoft 側で作った**参照データ**（合計 96 KB）が入っている。

参照データの中身:

| ファイル | 中身 | 何のために |
|---|---|---|
| `work/gofnt_chianti901.txt` | 22 輝線の G(T)（0.1 dex） | CHIANTI を落とさなくても DEM が解ける |
| `work/gofnt_chianti901_005.txt` | 同（0.05 dex） | demregpy 用 |
| `work/aia94_fe18_response.txt` | AIA 94 の Fe XVIII 応答 | 公式応答は低温線込みで使えない |
| `work/eis_calcurve_20110702.txt` | 打ち上げ後較正カーブ | 較正の効きを試す |
| `work/idl_intensities_tied.csv` | IDL で出した輝線強度 | 自分の結果の答え合わせ |
| `work/mcmc_dem_result.txt` | PINTofALE MCMC の DEM | 手法比較の相手 |

## ★ Colab の保存とセッションについて（最初に知っておくこと）

GitHub から開いたノートは、Colab では**読み取り専用の一時セッション**として扱われる。

| | どうなるか |
|---|---|
| ノートへの編集・実行結果 | **保存されない**。閉じたら消える |
| ダウンロードしたデータ (`data/`, `work/`) | **仮想マシンごと消える** |
| Google Drive | **何も書かれない**（自分でマウントしない限り触らない） |

**残したいときは「ファイル → ドライブにコピーを保存」。**
`MyDrive/Colab Notebooks/` にコピーが作られ、以降はそれが自分のノートになる
（ただし教材リポジトリ側が更新されても、そのコピーには反映されない）。

**仮想マシンが消えるタイミング**: 放置すると数十分で切断、
使っていても無料枠では最長で半日程度。
→ **EIS の 94 MB はセッションが切れるたびに落とし直し**になる。

そのため、**どのノートも「必要なデータをその場で取得する」設計**にしてある。
講習会の途中で切れても、先頭のセルから流し直せば復帰できる。

落とし直しを避けたい人は Drive をマウントして `data/` をそこに置いてもよい:

```python
from google.colab import drive
drive.mount('/content/drive')
```

（許可ダイアログを踏む必要があるので、講習会の既定にはしていない）

## 0-3. EIS のデータを取る

**NRL のアーカイブ**から level-1 HDF5 を直接落とす。ユーザ登録は要らない。

    https://eis.nrl.navy.mil/level1/hdf5/YYYY/MM/DD/eis_YYYYMMDD_HHMMSS.{data,head}.h5

- `.data.h5` (94 MB): スペクトルの中身
- `.head.h5` (421 KB): ヘッダ（波長軸、ポインティング、露出時間など）

**level-1 とは**: CCD の生データ（level-0）に対して
ペデスタル・暗電流を引き、不良画素と宇宙線を除き、
実効面積で割って物理単位 (erg cm⁻² s⁻¹ sr⁻¹ Å⁻¹) にしたもの。

In [ ]:
import os
os.makedirs("data/eis", exist_ok=True)
base = "https://eis.nrl.navy.mil/level1/hdf5/2011/07/02"
for f in ["eis_20110702_030712.data.h5", "eis_20110702_030712.head.h5"]:
    if not os.path.exists(f"data/eis/{f}"):
        !wget -q -c -P data/eis {base}/{f}
!ls -lh data/eis/

## 0-4. AIA の画像を取る

**JSOC の synoptic アーカイブ**を使う。1024×1024（2.4″/画素）、1 枚 1 MB 弱、
**登録不要**。

    http://jsoc.stanford.edu/data/aia/synoptic/YYYY/MM/DD/HHHH/AIAyyyymmdd_hhmm_wwww.fits

フルディスクの level-1（4096²、1 枚 65 MB）は VSO 経由で取れるが、
サーバが遅くてタイムアウトしやすい。**Colab では synoptic を使うこと。**
15″×23″ の箱なら 2.4″/画素でも 6×10 画素あり、平均値を出すには十分。

In [ ]:
os.makedirs("data/sdo/synoptic", exist_ok=True)
base = "http://jsoc.stanford.edu/data/aia/synoptic/2011/07/02/H0300"
for w in ["0094", "0171", "0193"]:
    f = f"AIA20110702_0338_{w}.fits"
    if not os.path.exists(f"data/sdo/synoptic/{f}"):
        !wget -q -c -P data/sdo/synoptic {base}/{f}
!ls -lh data/sdo/synoptic/

## 0-5. 動作確認

EIS のデータを 1 つ読んでみる。Fe XII 195.119 Å は活動領域で最も明るい輝線。

In [ ]:
cube = eispac.read_cube("data/eis/eis_20110702_030712.data.h5", 195.119)
print("データの形 (ny, nx, nwvl) =", cube.data.shape)
print("波長 [Å]:", float(cube.wavelength[0,0,0]), "-", float(cube.wavelength[0,0,-1]))
print("単位:", cube.unit)

### この観測がどういうものか

- **512 × 60 × 24** = (スリット方向の画素) × (ラスターのステップ) × (波長の画素)
- EIS は**スリット分光器**。細長いスリット（1″×512″）を太陽に当て、
  その 1 次元の像を波長分散させて CCD に落とす。
- 2 次元の画像がほしいので、**スリットを横に 60 回振る** → これがラスター。
- **★ 重要**: 1 ステップ約 60 秒なので、**全体で約 62 分かかる**。
  画像に見えるが**同時刻ではない**。左端と右端で 1 時間離れている。

In [ ]:
h = cube.meta["index"]
print("観測プログラム :", h["stud_acr"])          # スタディの略称
print("提案者         :", h["st_auth"])           # study author
print("開始           :", h["date_obs"])
print("終了           :", h["date_end"])
print("ラスター step数 :", h["nraster"])
print("露出時間 [s]   :", cube.meta["duration"][0])   # ステップごとに入っている
print("スリット       :", h["slit_id"])

観測プログラム名は `HPW021_VEL_120x512v1`、study author は `Harry Warren`。
**HPW = Harry P. Warren** — 論文の著者本人が設計した観測である。
「使いたい輝線が入った観測を自分で設計する」ところまで含めて研究になる。

**★ 露出時間はヘッダの `exptime` には入っていない**（`None` が返る）。
EIS はステップごとに露出時間を持ちうるので、
eispac は `cube.meta["duration"]`（長さ = ラスターのステップ数）に入れている。

---

## 0-6. スペクトルウィンドウを見る

EIS は 171–212 Å と 245–291 Å の 2 バンドを観測できるが、
全部を降ろすとテレメトリが足りない。そこで
**必要な輝線の周りだけを切り出して降ろす** → これが「スペクトルウィンドウ」。

**どの輝線が使えるかは観測プログラムの設計時に決まっている。**
この観測には 25 個のウィンドウがあり、論文が使う 22 輝線が全部入っている。

In [ ]:
from eispac import read_wininfo
wi = read_wininfo("data/eis/eis_20110702_030712.head.h5")
print(f"{'#':>3} {'line_id':<22} {'wvl_min':>9} {'wvl_max':>9}")
for w in wi:
    print(f"{w['iwin']:3d} {str(w['line_id']):<22} {w['wvl_min']:9.3f} {w['wvl_max']:9.3f}")

---

## まとめ

- パッケージはすべて pip で入る
- データは**登録不要**で取れる（EIS は NRL、AIA は JSOC synoptic）
- この観測は **62 分かけて撮った 60 ステップのラスター**
- **25 個のスペクトルウィンドウ**に論文の 22 輝線が全部入っている

次（モジュール 1）では、フィットせずに手早く強度マップを作り、
「どこを解析するか」を決める。

# モジュール 1: EIS のデータを見る

**所要時間 40 分**

**このノートで身につくこと**

1. EIS が撮っているのは**画像ではなくスペクトル**だと体で分かる
2. ラスター画像の x 軸が**空間であると同時に時間**であることを確認する
3. 輝線を変えると**同じ場所がまったく違う姿に見える**ことを見る
   → これが講習会全体の動機付け
4. 「どの輝線が使えるか」は観測プログラムの設計時に決まっていることを知る

前提: モジュール 0（環境構築とデータ取得）が終わっていること。

## 1-0. 準備

Colab で**このノートから始めた人**もここで動くように、
足りないファイルだけ取ってくる（既にあれば何もしない）。

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import eispac

EIS_FILE = "data/eis/eis_20110702_030712.data.h5"


def ensure(url, path):
    """path が無ければ url から落とす。あれば何もしない。"""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", url)
    urllib.request.urlretrieve(url, path)
    return path


base = "https://eis.nrl.navy.mil/level1/hdf5/2011/07/02"
for ext in ("data", "head"):
    ensure(f"{base}/eis_20110702_030712.{ext}.h5",
           f"data/eis/eis_20110702_030712.{ext}.h5")
print("ok")

## 1-1. データを 1 つ読む

`eispac.read_cube(ファイル, 波長)` で、その波長を含む
**スペクトルウィンドウ**を丸ごと読む。

In [ ]:
cube = eispac.read_cube(EIS_FILE, 195.119)      # Fe XII 195.119 Å
print("shape (y, x, wavelength) =", cube.data.shape)
print("単位                     =", cube.unit)

**`(512, 60, 24)` が意味するもの**

| 軸 | 数 | 正体 |
|---|---|---|
| 0 | 512 | **スリットに沿った空間**（1″/画素） |
| 1 | 60 | **ラスターのステップ**（2″/画素） |
| 2 | 24 | **波長**（0.0223 Å/画素） |

EIS は細長いスリット（1″×512″）を太陽に当て、その 1 次元の像を
波長分散させて CCD に落とす。**1 回の露出で得られるのは
(空間 512) × (波長) の 2 次元**であって、画像ではない。

2 次元の画像がほしければ**スリットを横に振る**。これがラスター。

## 1-2. まずスペクトルを 1 本見る

「EIS はスペクトルを撮っている」を実感するために、
1 画素分のスペクトルをそのまま描く。

In [ ]:
y, x = 250, 35          # 活動領域コアの中の 1 画素
plt.figure(figsize=(7, 4))
plt.plot(cube.wavelength[y, x, :], cube.data[y, x, :], "o-", ms=4)
plt.axvline(195.119, color="r", ls="--", lw=1, label="Fe XII 195.119")
plt.xlabel("wavelength [Å]")
plt.ylabel(f"intensity [{cube.unit}]")
plt.title(f"EIS spectrum, single pixel (y={y}, x={x})")
plt.legend()
plt.tight_layout()
plt.show()

**見えていること**

- 山が 1 つ。これが Fe XII 195.119 Å。**幅は 3–4 画素しかない**。
- 台が浮いている。これが背景（連続光＋散乱光）。
- **輝線強度はこのガウシアンの面積**。次のモジュールでこれを測る。

山の幅 σ ≈ 0.030 Å は**ほとんどが装置の幅**であって、
プラズマの熱運動や乱流速度はその上に乗るわずかな超過分。
だから線幅から速度を出すのは繊細な仕事になる。

## 1-3. ラスター画像を作る（フィット無しのクイックルック）

波長方向にただ足すだけ。連続光や隣の輝線も混ざるので**強度としては不正確**だが、
「どこに何があるか」を掴むには数秒で済むこの方法が便利。

**8 つの輝線で同じ場所を見る。** 並べる順は形成温度の順。

### ★ その前に: 欠損値の罠

単純に `np.nansum(cube.data, axis=2)` とやると**横に黒い縞**が出る。
まずそれを見てから、原因を確かめる。

In [ ]:
img_naive = np.nansum(cube.data, axis=2)        # 素直に足しただけ
plt.figure(figsize=(4, 8))
v = np.sqrt(np.clip(img_naive, 0, None))
plt.imshow(v, origin="lower", aspect="auto", cmap="inferno",
           vmin=0, vmax=np.nanpercentile(v, 99.5))
plt.title("np.nansum only → black stripes")
plt.xlabel("x [pix]"); plt.ylabel("y [pix]")
plt.tight_layout()
plt.show()

# 縞の正体を数字で確かめる
print("NaN の数        :", int(np.isnan(cube.data).sum()))      # → 0 個！
print("データの最小値  :", float(np.nanmin(cube.data)))         # → 大きな負の数
print("cube.mask で落とされるサンプル数:", int(np.asarray(cube.mask).sum()))

**★ 欠損値は NaN では入っていない。**

EIS の level-1 では、不良画素・宇宙線ヒットで捨てられたサンプルは
**大きな負のフラグ値**（level-0 の `-100` に較正係数を掛けたもの）として入っている。
だから

- `np.isnan` / `np.nansum` では**素通りする**
- 足すと大きな負の数が入り、その行だけ暗くなる → 黒い縞
- **エラーは一切出ない**

eispac はこれを `cube.mask`（True = 使ってはいけない）に立ててくれるので、
**必ずこれで落とす**。この先の全モジュールで効いてくる:

- 箱の中で平均するとき、欠損を入れると強度が静かに下がる（モジュール 2）。
  論文 §3 もわざわざ *"In computing these averaged profiles, missing data are
  not included"* と書いている。
- この教材の準備でも、マスクを忘れたまま解析していて、
  弱い線で数 %（Ca XVII +6.9%、Ca XVI −4.6%、Ar XIV +3.2%）ずれていた。

In [ ]:
def raster_image(datafile, wvl):
    """波長方向に積んで強度マップにする。欠損サンプルは平均に入れない。"""
    c = eispac.read_cube(datafile, wvl)
    d = np.where(np.asarray(c.mask, dtype=bool), np.nan, c.data)
    return np.nanmean(d, axis=2) * d.shape[2]      # 平均 × サンプル数 = 積分値

img = raster_image(EIS_FILE, 195.119)
plt.figure(figsize=(4, 8))
v = np.sqrt(np.clip(img, 0, None))
plt.imshow(v, origin="lower", aspect="auto", cmap="inferno",
           vmin=0, vmax=np.nanpercentile(v, 99.5))
plt.title("with cube.mask → stripes gone")
plt.xlabel("x [pix]"); plt.ylabel("y [pix]")
plt.tight_layout()
plt.show()

### 8 つの輝線で並べる

In [ ]:
PANELS = [
    (275.368, "Si VII 275.4",  "0.6 MK  moss"),
    (184.536, "Fe X 184.5",    "1.1 MK"),
    (195.119, "Fe XII 195.1",  "1.6 MK"),
    (202.044, "Fe XIII 202.0", "1.8 MK"),
    (262.984, "Fe XVI 263.0",  "2.8 MK"),
    (193.874, "Ca XIV 193.9",  "3.5 MK"),
    (200.972, "Ca XV 201.0",   "4.5 MK"),
    (192.858, "Ca XVII 192.9", "5.6 MK  (blended)"),
]

ext = cube.meta["extent_arcsec"]        # [x0, x1, y0, y1] （太陽面座標, arcsec）
fig, axes = plt.subplots(1, len(PANELS), figsize=(2.3 * len(PANELS), 9))
for ax, (wvl, label, temp) in zip(axes, PANELS):
    v = np.sqrt(np.clip(raster_image(EIS_FILE, wvl), 0, None))   # 平方根で暗部を持ち上げる
    lo, hi = np.nanpercentile(v, [1, 99.5])
    ax.imshow(v, origin="lower", extent=ext, aspect="equal",
              cmap="inferno", vmin=lo, vmax=hi)
    ax.set_title(f"{label}\n{temp}", fontsize=9)
    ax.set_xlabel("Solar X [″]")
    if ax is not axes[0]:
        ax.set_yticklabels([])
axes[0].set_ylabel("Solar Y [″]")
fig.suptitle("NOAA 1243   2011-07-02 03:07 UT   (same field of view, 8 spectral lines)",
             fontsize=12)
fig.tight_layout(rect=[0, 0.01, 1, 0.965])      # suptitle と重ならないように
plt.show()

**図の中の英語について**: Colab には日本語フォントが入っていないので、
図のラベルを日本語にすると豆腐（□）になる。
この教材では**図は英語、説明は日本語**で統一している。
どうしても図に日本語を入れたければ `!pip install japanize-matplotlib` を使う。

## 1-4. ★ ここが講習会全体の動機

**同じ場所なのに、輝線を変えると別物に見える。**

| 温度 | 見えるもの |
|---|---|
| Si VII (0.6 MK) | **まだら模様** = moss。高温ループの**足元**が遷移層で光っている |
| Fe X–XIII (1–2 MK) | **細いループ**が何本も見える。周辺部まで広がる |
| Fe XVI 以上 (2.8 MK–) | ループが消え、**中心部の塊**だけが残る = 活動領域コア |

コロナが単一温度なら、どの輝線で撮っても同じ絵になるはずである。
そうならないということは、**視線上に色々な温度のプラズマが混ざっている**。
その温度ごとの量を測るのが **DEM 解析**（モジュール 6, 7）。

**★ Ca XVII のパネルをよく見る。** 5.6 MK の線なのに
Fe XVI や Ca XV（もっと低温）よりも Fe XII (1.6 MK) に似ていないか？
→ **ブレンドしている**（Fe XI と O V が混ざっている）。モジュール 8 で解く。

**★ 輝線によって残る筋の位置が違う**のにも注意。
ウィンドウごとに CCD の別の領域を使うので、不良画素の位置も輝線ごとに違う。
「1 本の線でうまくいったから全部大丈夫」とはならない。

## 1-5. ★ この「画像」は同時刻ではない

ラスターは**スリットを 1 ステップずつ動かして**作る。
つまり x 軸は空間であると同時に**時間軸**でもある。

In [ ]:
h = cube.meta["index"]
dur = cube.meta["duration"]             # ステップごとの露出時間 [s]
print("観測プログラム :", h["stud_acr"], " (提案者:", h["st_auth"] + ")")
print("開始           :", h["date_obs"])
print("終了           :", h["date_end"])
print("ステップ数     :", h["nraster"])
print("露出時間 [s]   :", f"{dur[0]:.1f}  (1 ステップあたり)")
print()
from astropy.time import Time
t0, t1 = Time(h["date_obs"]), Time(h["date_end"])
total = (t1 - t0).to_value("s")
print(f"全体の所要時間 : {total/60:.1f} 分")
print(f"1 ステップ     : {total/h['nraster']:.1f} 秒  "
      f"(= 露出 {dur[0]:.0f} 秒 + 読み出し等)")
print(f"視野           : {h['fovx']:.1f}″ x {h['fovy']:.1f}″")
print(f"x 方向のサンプリング : {h['fovx']/h['nraster']:.2f}″/step  "
      f"(スリット幅は {h['slit_id']})")

**62 分かけて撮っている。** 左端と右端では 1 時間離れている。

帰結（受講者が必ず引っかかるところ）:

- **時間変化する現象には使えない。** フレアやジェットがラスターの途中で
  起きると、その列だけ別の状態が写る。
- 「速度マップ」を作っても、それは同時刻の速度場ではない。
- 一方、活動領域コアの**定常的な**構造を測る今回の目的には問題ない。
  むしろ 62 分の平均になるので S/N の面では有利。

**★ AIA (12 秒カデンス) と重ねるときは、EIS のどの列がどの時刻かを意識する。**
我々は AIA 03:38 UT（ラスターのほぼ中央の時刻）を使う。

## 1-6. どの輝線が使えるかは、観測の設計時に決まっている

EIS は 171–212 Å と 245–291 Å を観測できるが、
**全波長を降ろすとテレメトリが足りない**。
そこで必要な輝線の周りだけを切り出して降ろす → **スペクトルウィンドウ**。

In [ ]:
wi = eispac.read_wininfo(EIS_FILE.replace(".data.h5", ".head.h5"))
print(f"{'#':>3} {'line_id':<22} {'wvl_min':>9} {'wvl_max':>9} {'幅[Å]':>7}")
for w in wi:
    print(f"{w['iwin']:3d} {str(w['line_id']):<22} "
          f"{w['wvl_min']:9.3f} {w['wvl_max']:9.3f} {w['wvl_max']-w['wvl_min']:7.3f}")
print(f"\n{len(wi)} ウィンドウ")

**読み方**

- ウィンドウ名（`line_id`）は**目安**であって、そこに入っている輝線が
  1 本とは限らない。`CA XVII 192.470` のウィンドウは 1.2 Å と広く、
  Fe XI・O V・Ca XVII が全部入っている（だからブレンドが解ける）。
- **Ca XIV / Ca XV / Ca XVI / Ca XVII が入っているのが決定的に重要**。
  これらが 3 MK 以上を拘束する。入っていない観測では
  「コアの高温プラズマ」の議論ができない。
- `FE XXIV 255.000` はフレア用。今回は使わない。

論文 Table 1 の 15 活動領域のうち **region 1 (2010-06-19) には Ca XIV–XVI が無い**。
同じ解析ができない例として、モジュール 10 で扱う。

→ **自分の科学のために EIS を使うときは、まず
  「その観測プログラムに必要な線が入っているか」を確認する。**

## 1-7. 演習

1. `PANELS` に自分で輝線を足して描いてみる。
   使える波長は 1-6 のウィンドウ一覧から選ぶ
   （例: `186.750` Fe XII、`276.300` Mg V、`278.400` Mg VII）。
2. 1-2 のスペクトルを、**moss の上**（例 `y=250, x=10` 付近）と
   **コアの中**（`y=250, x=35`）で描き比べる。
   Si VII 275.368 のウィンドウでやると差が大きい。
3. `Ca XVII 192.9` のパネルが Fe XII に似ている理由を、
   1-6 のウィンドウ幅の表から説明してみる。

## まとめ

- EIS が撮るのは **(空間 512) × (波長)**。画像はスリットを振って自分で作る
- **x 軸は時間軸でもある**（この観測は 62 分）
- **輝線ごとに違う温度が見える** → これを定量するのが DEM 解析
- **使える輝線は観測プログラムで決まっている**

次（モジュール 2）では、この山にガウシアンを当てて
**輝線強度という数値**を取り出す。

# モジュール 2: スペクトル線をフィットして「強度」を取り出す

**所要時間 50 分**

**このノートで身につくこと**

1. 輝線強度とは**ガウシアンの面積**であり、測定値ではなく**フィットの産物**だと分かる
2. eispac のテンプレート機構を使いこなす（`parinfo`, `line_ids`, `tied`）
3. **成分の順番の罠**を自分で踏んで確認する（受講者が最も高確率で間違えるところ）
4. 論文の手順 = **「箱の中で平均してからフィット」**の理由を、速度と S/N の両面で理解する
5. 誤差の入れ方を知る（統計誤差だけでは χ² が発散する）

前提: モジュール 1。

In [ ]:
import os
import time
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import eispac

EIS_FILE = "data/eis/eis_20110702_030712.data.h5"


def ensure(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", url)
    urllib.request.urlretrieve(url, path)
    return path


base = "https://eis.nrl.navy.mil/level1/hdf5/2011/07/02"
for ext in ("data", "head"):
    ensure(f"{base}/eis_20110702_030712.{ext}.h5",
           f"data/eis/eis_20110702_030712.{ext}.h5")
print("ok")

## 2-1. 何を測るのか

EIS が返すのは各波長画素の強度。そこから輝線強度を出すには
**ガウシアン + 背景**を当てて、ガウシアンの面積を取る:

$$ I(\lambda) = A \exp\left[-\frac{(\lambda-\lambda_0)^2}{2\sigma^2}\right] + b $$

$$ I_{\rm line} = \int A\,e^{-(\lambda-\lambda_0)^2/2\sigma^2} d\lambda
   = A\,\sigma\sqrt{2\pi} $$

得られる 3 つの量:

| パラメータ | 物理量 |
|---|---|
| 面積 $A\sigma\sqrt{2\pi}$ | **輝線強度** [erg cm⁻² s⁻¹ sr⁻¹] → DEM に使う |
| 中心 $\lambda_0$ | **ドップラー速度** |
| 幅 $\sigma$ | 熱運動 + 非熱的速度（ただし装置幅が支配的） |

**★ 「強度」は測定値ではない。** 背景をどこに引くか、隣の線をどう扱うか、
ガウシアン何本を当てるか——**全部モデルの仮定**である。
論文の値と自分の値が違うとき、まずここを疑う。

## 2-2. eispac のテンプレート

eispac には輝線ごとのフィット設定（**テンプレート**）が同梱されている。

In [ ]:
path = eispac.data.get_fit_template_filepath("fe_12_195_119.2c.template.h5")
tmplt = eispac.read_template(path)
print("ファイル:", os.path.basename(path))
print("line_ids:", tmplt.template["line_ids"])
print("n_gauss :", tmplt.template["n_gauss"], "  n_poly:", tmplt.template["n_poly"])
print()
print(f"{'#':>2} {'value':>12} {'fixed':>6} {'limited':>10} {'limits':>22} {'tied':>8}")
for i, p in enumerate(tmplt.parinfo):
    print(f"{i:2d} {p['value']:12.4f} {p['fixed']:6d} {str(p['limited']):>10} "
          f"{str(np.round(p['limits'], 3)):>22} {str(p['tied']):>8}")

**パラメータの並びは `[振幅, 中心, 幅] × ガウシアンの本数 + 背景の係数`**。
`2c` テンプレートは 2 成分なので 3×2 + 1 = **7 パラメータ**。

- `.1c` = 1 成分、`.2c` = 2 成分、`.3c` = 3 成分
- `limited` / `limits` で範囲を縛る（振幅は正、中心は ±0.1 Å など）
- **`tied`** は「他のパラメータに縛る」しくみ。上の出力では第 2 成分の
  中心が `p[1]+0.06`（第 1 成分から 0.06 Å 離れた位置）に、
  幅が `p[2]`（第 1 成分と同じ）に縛られている。
  つまり **2 本目は原子データで位置を決め打ちしている**。
  この機構がブレンドを解くときの主役になる（モジュール 8）。

Fe XII 195.119 に**なぜ 2 成分**必要か: 195.179 Å に弱い Fe XII の線があり、
密度が高いと無視できない。1 成分で当てると強度が数 % 過大になる。

## 2-3. ★ 罠: 成分の順番は波長順とは限らない

多成分テンプレートで**自分がほしい線が第 0 成分とは限らない**。
論文の 22 輝線のうち **4 本**がこれに該当する。必ず `line_ids` で確認すること。

In [ ]:
for name in ["fe_12_195_119.2c", "fe_13_203_826.2c", "fe_14_270_519.2c",
             "ar_14_194_396.2c", "ca_14_193_874.2c"]:
    t = eispac.read_template(eispac.data.get_fit_template_filepath(name + ".template.h5"))
    ids = [str(s) for s in t.template["line_ids"]]
    print(f"{name:<20} {ids}")

`fe_13_203_826.2c` の**第 0 成分は Fe XII 203.720** であって、
ほしい Fe XIII 203.826 は第 1 成分。`component=0` と書いたら
**別のイオンの強度**を DEM に入れてしまう。しかもエラーは出ない。

手で書くと必ず間違えるので、**波長で自動照合する**関数を使う。

In [ ]:
def pick_component(template, target_wvl):
    """line_ids を見て、目的波長に一番近いガウシアン成分の番号を返す。"""
    ids = [str(s) for s in template.template["line_ids"]]
    best, bestd = 0, 1e9
    for i, s in enumerate(ids):
        try:
            w = float(s.split()[-1])
        except ValueError:
            continue
        if abs(w - target_wvl) < bestd:
            best, bestd = i, abs(w - target_wvl)
    return best, ids


t = eispac.read_template(eispac.data.get_fit_template_filepath(
    "fe_13_203_826.2c.template.h5"))
print(pick_component(t, 203.826))      # → (1, [...]) になるはず

## 2-4. まず素直に「全ラスターをフィット」してみる

eispac の標準的な使い方。ただし **1 輝線 30720 スペクトルで 2-3 分**かかる。
22 輝線なら 1 時間。ここでは**活動領域の部分だけ**（y = 200–320）を試す。

In [ ]:
tmplt = eispac.read_template(eispac.data.get_fit_template_filepath(
    "fe_12_195_119.2c.template.h5"))
cube = eispac.read_cube(EIS_FILE, tmplt.central_wave)

t0 = time.time()
fit = eispac.fit_spectra(cube[200:320, :, :], tmplt, ncpu=2, ignore_warnings=True)
dt = time.time() - t0
n = 120 * 60
print(f"\n{n} スペクトルを {dt:.0f} 秒でフィット "
      f"({1000*dt/n:.1f} ms/スペクトル)")
print(f"→ 全ラスター (512x60) なら約 {dt*512/120/60:.1f} 分、22 輝線なら約 "
      f"{dt*512/120*22/60:.0f} 分")

In [ ]:
m_int = fit.get_map(component=0, measurement="intensity")
m_vel = fit.get_map(component=0, measurement="velocity")

fig, axes = plt.subplots(1, 2, figsize=(9, 7))
d = np.sqrt(np.clip(m_int.data, 0, None))
axes[0].imshow(d, origin="lower", aspect="auto", cmap="inferno",
               vmin=0, vmax=np.nanpercentile(d, 99.5))
axes[0].set_title("Fe XII 195.119  intensity")
axes[1].imshow(m_vel.data, origin="lower", aspect="auto", cmap="RdBu_r",
               vmin=-20, vmax=20)
axes[1].set_title("Doppler velocity [km/s]")
for ax in axes:
    ax.set_xlabel("x [pix]")
axes[0].set_ylabel("y [pix]  (200-320 の範囲)")
fig.tight_layout()
plt.show()

速度マップに構造が見える（青 = 上昇流）。これはこれで面白いが、
**論文がやりたいのは 22 輝線の強度**であって、この速度ではない。
22 輝線を全ラスターでフィットするのは時間の無駄になる。

## 2-5. 論文の手順: **箱の中で平均してからフィットする**

論文 §3 はこう書いている:

> we extract the EIS data from each spectral window in the selected field of view
> and **average them together** (missing data are not included in the average)
> to form high signal-to-noise line profiles, which are then fit with
> single Gaussians

**順番が「平均 → フィット」である**ことが重要。理由は 2 つ:

1. **速い**。240 画素を平均すれば、フィットするのは 1 本のプロファイルだけ。
   22 輝線でも数秒で終わる（Colab で決定的）。
2. **S/N が上がる**。弱い線（Ca XVI は最も明るい線の 1/200）は
   1 画素では埋もれている。平均して初めてフィットできる。

逆順（各画素をフィットしてから平均）とは**同じにならない**。
フィットは非線形なので、平均と可換ではない。
論文がどちらをやったかを読み取ることが、再現の前提になる。

使う箱は **モジュール 4 で選ぶ**。ここでは結果を先取りして使う。

In [ ]:
BOX = dict(y0=244, y1=274, x0=32, x1=40)     # モジュール 4 で決めた inter-moss 箱


def average_spectrum(datafile, wvl, y0, y1, x0, x1):
    """箱の中でスペクトルを平均する。欠損サンプルは平均に入れない。

    ★ 欠損は NaN ではなく**大きな負のフラグ値**（モジュール 1 参照）。
      eispac が立てる cube.mask で必ず落とす。
    """
    c = eispac.read_cube(datafile, wvl)
    data = c.data[y0:y1, x0:x1, :]
    errs = c.uncertainty.array[y0:y1, x0:x1, :]
    wave = c.wavelength[y0:y1, x0:x1, :]
    bad = np.asarray(c.mask[y0:y1, x0:x1, :], dtype=bool)

    good = np.isfinite(data) & ~bad
    n = good.sum(axis=(0, 1))                            # 波長ごとの有効画素数
    inten = np.nansum(np.where(good, data, 0), axis=(0, 1)) / np.maximum(n, 1)
    # 平均の誤差 = sqrt(Σσ²)/N
    sig = np.sqrt(np.nansum(np.where(good, errs**2, 0), axis=(0, 1))) / np.maximum(n, 1)
    inten[n == 0], sig[n == 0] = np.nan, np.nan
    return np.nanmean(wave, axis=(0, 1)), inten, sig, int(np.median(n))


wave, inten, sig, npix = average_spectrum(EIS_FILE, 195.119, **BOX)
print(f"箱 y=[{BOX['y0']}:{BOX['y1']}] x=[{BOX['x0']}:{BOX['x1']}] "
      f"= {(BOX['y1']-BOX['y0'])*(BOX['x1']-BOX['x0'])} 画素")
print(f"平均に使えた画素数（中央値）: {npix}")
print(f"ピーク強度 {np.nanmax(inten):.0f} ± {sig[np.nanargmax(inten)]:.1f}  "
      f"→ 相対誤差 {100*sig[np.nanargmax(inten)]/np.nanmax(inten):.2f}%")

In [ ]:
t0 = time.time()
fit1 = eispac.fit_spectra(inten, tmplt, wave=wave, errs=sig, ncpu=1,
                          ignore_warnings=True)
print(f"1 本のプロファイルのフィット: {1000*(time.time()-t0):.0f} ms")

wfit, pfit = fit1.get_fit_profile()          # 細かい波長グリッドでのモデル曲線
plt.figure(figsize=(7, 4.5))
plt.errorbar(wave, inten, yerr=sig, fmt="o", ms=4, label="box-averaged data")
plt.plot(np.ravel(wfit), np.ravel(pfit), "-", lw=2, label="fit (2 Gaussians + bg)")
plt.axvline(195.119, color="r", ls="--", lw=1)
plt.axvline(195.179, color="g", ls="--", lw=1)
plt.xlabel("wavelength [Å]")
plt.ylabel("intensity")
plt.title("Fe XII 195.119, averaged over the box")
plt.legend()
plt.tight_layout()
plt.show()

## 2-6. 強度がガウシアンの面積であることを数値で確かめる

In [ ]:
p = np.atleast_1d(fit1.fit["params"]).ravel()
A, lam0, sigma = p[0], p[1], p[2]
I_fit = float(np.atleast_1d(fit1.fit["int"][..., 0]).ravel()[0])
print(f"A     = {A:12.2f}")
print(f"λ0    = {lam0:12.4f} Å")
print(f"σ     = {sigma:12.4f} Å")
print(f"A σ √(2π) = {A*sigma*np.sqrt(2*np.pi):10.2f}")
print(f"eispac の int = {I_fit:10.2f}   ← 一致する")
print()
print(f"背景 b = {p[-1]:.1f}  （ピークの {100*p[-1]/A:.0f}%）")
print(f"論文 Table 2 の Fe XII 195.119 = 1147.35 → 比 {I_fit/1147.35:.3f}")

**σ = 0.028 Å の意味**: EIS の装置幅は σ ≈ 0.027 Å（1″ スリット）。
つまり測っている幅の**ほとんどが装置由来**で、
プラズマの熱運動 + 非熱的速度はその上に乗るわずかな超過分。
線幅から速度を出すには装置幅を正確に知る必要がある（今回は使わない）。

## 2-7. ★ 誤差の入れ方（ここを間違えると DEM で破綻する）

上で見たとおり、240 画素を平均した後の**統計誤差は 0.2%** しかない。
一方、論文が使っている誤差は **22%**。

In [ ]:
chi2 = float(np.atleast_1d(fit1.fit["chi2"]).ravel()[0])
print(f"このフィットの χ² = {chi2:.0f}   (データ点 {len(wave)}, パラメータ 7)")
print(f"χ²_red = {chi2/(len(wave)-7):.0f}   ← 1 のはずが桁違い")

**なぜ χ² が桁違いに大きいのか**

統計誤差だけを使うと、誤差が小さすぎて「ガウシアン + 直線背景」という
**モデルのわずかな不完全さ**が全部 χ² に化ける。
実際のスペクトルには弱いブレンドや非対称性があり、
0.2% の精度でガウシアンに一致することはない。

**論文の 22% はどこから来るか**

- EIS の**絶対較正**の不確かさ（打ち上げ前較正で ~20%、劣化補正でさらに増える）
- これは**系統誤差**なので、画素を平均しても減らない

→ DEM を解くときは **σ_I = max(統計誤差, 0.22 × I)** のように
  **較正誤差を床として入れる**。これをやらないと χ² が発散して
  DEM インバージョンが収束しない（モジュール 7 で実際に見る）。

**★ 教訓**: 「誤差が小さい」ことは良いことではない。
**何の誤差を見積もっているか**を意識する。

## 2-8. 22 輝線をまとめてフィットする

論文 Table 2 の 22 輝線と、対応する eispac テンプレートの表は
リポジトリの `scripts/lines_warren2012.py` にある。

In [ ]:
import sys
sys.path.insert(0, "scripts")
from lines_warren2012 import LINES        # (ion, 波長, テンプレート, 論文値, 論文σ)

print(f"{'line':<16}{'template':<26}{'I_paper':>9}")
for ion, wvl, tname, ip, sp in LINES[:5]:
    print(f"{ion+' '+f'{wvl:.3f}':<16}{tname.replace('.template.h5',''):<26}{ip:9.2f}")
print(f"... 全 {len(LINES)} 本")

In [ ]:
t0 = time.time()
rows = []
for ion, wvl, tname, i_paper, sig_paper in LINES:
    w, I, s, npix = average_spectrum(EIS_FILE, wvl, **BOX)
    t = eispac.read_template(eispac.data.get_fit_template_filepath(tname))
    comp, ids = pick_component(t, wvl)
    f = eispac.fit_spectra(I, t, wave=w, errs=s, ncpu=1, ignore_warnings=True)
    i_fit = float(np.atleast_1d(f.fit["int"][..., comp]).ravel()[0])
    rows.append((ion, wvl, i_fit, i_paper, i_fit / i_paper, comp, ids[comp]))
print(f"\n22 輝線を {time.time()-t0:.0f} 秒でフィットした\n")

print(f"{'line':<16}{'I_fit':>10}{'I_paper':>10}{'ratio':>7}  component")
for ion, wvl, i_fit, i_paper, r, comp, cid in rows:
    print(f"{ion+' '+f'{wvl:.3f}':<16}{i_fit:10.2f}{i_paper:10.2f}{r:7.2f}  [{comp}] {cid}")

In [ ]:
import csv
os.makedirs("work", exist_ok=True)
with open("work/box_intensities.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["ion", "wvl", "I_fit", "I_paper", "ratio"])
    w.writerows([(r[0], r[1], r[2], r[3], r[4]) for r in rows])
print("wrote work/box_intensities.csv  （モジュール 5, 6, 7 で使う）")

ここまでで**論文 Table 2 と比べられる数字**が出た。
中身の議論（どれが合っていて、どれが合わないか）は**モジュール 5** で行う。

ここで先に 1 つだけ言っておくと、**Ca XVII 192.858 の ratio が 5 前後**に
なっているはずである。これはフィットの失敗ではなく
**ブレンド**（Fe XI と O V が混ざっている）で、モジュール 8 で解く。

同じ処理は `scripts/fit_box_spectra.py` にまとめてある:

```bash
python scripts/fit_box_spectra.py data/eis/eis_20110702_030712.data.h5 244 274 32 40
```

## 2-9. 演習

1. **成分の順番を間違えるとどうなるか**を体験する。
   上のループで `comp` を `0` に固定して Fe XIII 203.826 の値を見る。
   論文値との比がどう変わるか。
2. **マスクを外すとどうなるか**。`average_spectrum` の `& ~bad` を消して
   22 輝線を再実行し、どの線が何 % 変わるか調べる
   （ヒント: 弱い線ほど効く。Ca XVII、Ca XVI、Ar XIV）。
3. **箱の大きさを変える**。`BOX` の y 範囲を 2 倍にして、
   統計誤差と ratio がどう変わるか。誤差は減るが ratio は良くなるか？
4. 2-5 の逆順（各画素をフィットしてから平均）を Fe XII で試して、
   「平均 → フィット」との差を測る（`scripts/fit_perpixel_box.py` に実装がある）。

## まとめ

- 輝線強度は**ガウシアンの面積** $A\sigma\sqrt{2\pi}$。**フィットの産物**である
- **成分の順番は波長順ではない**。必ず `line_ids` で照合する
- 論文の手順は **「箱で平均 → フィット」**。速さと S/N の両方の理由がある
- **統計誤差だけでは足りない**。較正の系統誤差 22% を床として入れる

次（モジュール 3）では、EIS では測れない 7 MK のプラズマを
**AIA 94 Å から取り出す**。

# モジュール 3: AIA 94 Å から Fe XVIII (7 MK) を取り出す

**所要時間 50 分**

**このノートで身につくこと**

1. **なぜ EIS だけでは足りないか**（EIS の最高温は Ca XVII の ~5 MK）
2. AIA 94 Å の Fe XVIII を経験式で分離する（論文 Appendix）
3. ★ **論文に印刷された式の指数が誤植である**ことを、実データで自分で確かめる
4. moss（171 Å）と高温ループ（Fe XVIII）の空間分布の違いを見る
   → 次のモジュールで箱を選ぶための下準備

前提: モジュール 1, 2。EIS のデータは使わないので単独でも動く。

## 3-0. データを取る（登録不要）

**JSOC の synoptic アーカイブ**を使う。1024×1024（2.4″/画素）、1 枚 1 MB 弱。

    http://jsoc.stanford.edu/data/aia/synoptic/YYYY/MM/DD/HHHH/AIAyyyymmdd_hhmm_wwww.fits

- フルディスク level-1（4096²、1 枚 65 MB）は VSO 経由で取れるが遅い。
  **Colab では synoptic を使う。**
- synoptic は既に **level-1.5**（`lvl_num=1.5`）なので `aiapy` の
  `register` / `update_pointing` は不要。読んですぐ使える。
- 時刻 **03:38 UT** は EIS ラスター（03:07–04:09）のほぼ中央。

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import sunpy.map
from astropy.coordinates import SkyCoord

AIA_DIR = "data/sdo/synoptic"
AIA_BASE = "http://jsoc.stanford.edu/data/aia/synoptic/2011/07/02/H0300"


def ensure(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", os.path.basename(path))
    urllib.request.urlretrieve(url, path)
    return path


def load_aia(wave):
    """synoptic の AIA を読んで DN/s にする。"""
    f = ensure(f"{AIA_BASE}/AIA20110702_0338_{wave:04d}.fits",
               f"{AIA_DIR}/AIA20110702_0338_{wave:04d}.fits")
    m = sunpy.map.Map(f)
    exp = m.meta["exptime"]
    print(f"AIA {wave:4d}: {m.data.shape}  exptime={exp:.3f}s  "
          f"lvl={m.meta.get('lvl_num')}  {m.scale[0]:.2f}")
    return sunpy.map.Map(m.data / exp, m.meta)      # ★ 露光時間で割る


a94, a171, a193 = load_aia(94), load_aia(171), load_aia(193)

## 3-1. なぜ AIA が要るのか

EIS で観測できる**最高温の強い輝線は Ca XVII 192.858（~5 MK）**。
それより上を拘束するものが無いと、EM 分布の高温側の裾が決まらない。

しかも Ca XVII はブレンドしていて扱いが難しい（モジュール 8）。
**7 MK に効く独立な測定**がほしい。

そこで **AIA 94 Å の Fe XVIII 93.932 Å**（形成温度 log T ≈ 6.85 = 7 MK）を使う。

**問題**: 94 Å チャンネルには Fe XVIII 以外に
**Fe X 94.012 Å をはじめとする低温の線**が入っている。
静穏領域やループの足元では、94 Å の信号の**ほとんどが低温成分**。
そのまま使うと「7 MK のプラズマが大量にある」ことになってしまう。

## 3-2. 論文 Appendix の経験式

論文は 171 Å と 193 Å の混合から「warm 成分」を経験的に見積もって引く:

$$ x = \frac{f\,I_{171} + (1-f)\,I_{193}}{116.54}, \qquad f = 0.31 $$
$$ I_{94}^{\rm warm} = 0.39 \sum_i a_i x^{?}, \qquad
   a = [-7.31\times10^{-2},\ 9.75\times10^{-1},\ 9.90\times10^{-2},\ -2.84\times10^{-3}] $$
$$ I_{\rm FeXVIII} = I_{94} - I_{94}^{\rm warm} $$

較正の元データは 2010-03-22 12–13 UT の 1 時間平均（明るい輝点＋静穏領域）で、
規格化定数 116.54 と 0.39 は**そのときの中央値**。
つまり「**中央値の場所では x = 1、warm 成分 = 0.39 DN/s**」という約束になっている。

**★ 問題は指数 `?` である。** 論文の印刷は

$$ I_{94}^{\rm warm} = 0.39 \sum_{i=1}^{4} a_i x^{i} $$

だが、この字面どおりだと定数項が無い（x=0 で 0）。
一方、定数項つきの 3 次式（$a_i x^{i-1}$）という読み方もできる。
**どちらが正しいかは、実データを通してみれば決まる。**

## 3-3. ★ 誤植を自分で確かめる

2 つの読み方それぞれで warm 成分を計算し、**観測された 94 Å の強度**と比べる。
warm 成分は 94 Å の**一部**なのだから、観測値を超えたらその読み方は誤り。

In [ ]:
A = [-7.31e-2, 9.75e-1, 9.90e-2, -2.84e-3]
F_MIX, NORM_COMPOSITE, NORM_94, X_MAX = 0.31, 116.54, 0.39, 30.0

x = (F_MIX * a171.data + (1 - F_MIX) * a193.data) / NORM_COMPOSITE
x = np.clip(x, 0.0, X_MAX)          # 論文の指示どおり x は 30 で頭打ち

warm_printed = NORM_94 * sum(a * x**(i + 1) for i, a in enumerate(A))   # a_i x^i
warm_fixed = NORM_94 * sum(a * x**i for i, a in enumerate(A))           # a_i x^(i-1)

print(f"{'x の範囲':>12} {'観測 I_94':>10} {'印刷どおり':>12} {'定数項つき':>12}  {'画素数':>8}")
for lo, hi in [(0.5, 1.5), (2, 3), (5, 8), (12, 18), (25, 30)]:
    sel = (x >= lo) & (x < hi)
    if sel.sum() == 0:
        continue
    print(f"{lo:5.1f} - {hi:4.1f} {np.median(a94.data[sel]):10.2f} "
          f"{np.median(warm_printed[sel]):12.2f} {np.median(warm_fixed[sel]):12.2f}"
          f" {sel.sum():9d}")

**結論は一目瞭然**: 印刷どおりの読み方だと、warm 成分が
**観測された 94 Å の全強度を桁違いに超える**。物理的にありえない。
定数項つきの 3 次式なら観測値の少し下に収まる。

→ **論文 Eq.(A1) の指数は組版上の誤りで、正しくは $a_i x^{i-1}$**（定数項つき）。

**★ 注意**: x = 1（較正の中央値）では**どちらの読み方も 0.389 になる**。
論文が明記している唯一の数値がここなので、**この点だけでは判別できない**。
実データを広い x の範囲で通して初めて決まる。

**教訓**: 「論文に書いてある式」でも、**数値を通すまで信じない**。
特に指数・添字・単位は組版で落ちやすい。

In [ ]:
# 差し引きの引き算そのもの
def fe18(i94, i171, i193, f=F_MIX):
    xx = np.clip((f * np.asarray(i171, float) + (1 - f) * np.asarray(i193, float))
                 / NORM_COMPOSITE, 0.0, X_MAX)
    warm = NORM_94 * sum(a * xx**i for i, a in enumerate(A))
    return np.asarray(i94, float) - warm


# 多項式が x=30 で頭打ちにされている理由も確かめておく
xs = np.linspace(0, 40, 4001)
poly = sum(a * xs**i for i, a in enumerate(A))
print(f"多項式の極大は x = {xs[np.argmax(poly)]:.1f}")
print("→ 論文が x を 30 で頭打ちにしているのは、この折り返しの手前で止めるため")
print(f"x=1 での warm = {NORM_94*sum(a*1.0**i for i, a in enumerate(A)):.3f} DN/s "
      f"（論文の中央値 0.39 と一致）")

## 3-4. Fe XVIII マップを作る

活動領域の周りだけ切り出して 4 枚並べる。

In [ ]:
XC, YC = -330, 200          # NOAA 1243 のだいたいの位置 [arcsec]
HALF = 200 * u.arcsec

bl = SkyCoord((XC * u.arcsec) - HALF, (YC * u.arcsec) - HALF, frame=a94.coordinate_frame)
tr = SkyCoord((XC * u.arcsec) + HALF, (YC * u.arcsec) + HALF, frame=a94.coordinate_frame)

# ★ 罠: 同じ SkyCoord で切り出しても、波長ごとに配列サイズが 1 画素ずれることがある
for m, w in [(a94, 94), (a171, 171), (a193, 193)]:
    print(f"AIA {w:4d} を SkyCoord で切り出すと {m.submap(bl, top_right=tr).data.shape}")

**サイズが揃わない。** 引き算しようとすると
`operands could not be broadcast together` で落ちる。

原因は観測者距離 `dsun` がチャンネルごとに僅かに違い、
arcsec → 画素の換算が 1 画素未満ずれて、切り上げ・切り捨ての境目に乗ること。

**対処は 2 つ。今回のデータではどちらでもよいが、意味が違う。**

1. **同じ画素範囲で切る**（今回の synoptic は 3 波長とも
   `crpix/crval/cdelt/crota` が完全に一致しているので、これで厳密に正しい）
2. **`reproject_to()` で揃える**（WCS が違う場合はこちらが必須。
   フルディスク level-1 を自分で `aiapy.calibrate.register` した場合など)

**★ 確認せずに引き算しないこと。** 揃っていない画像を引くと、
構造の縁に沿って偽の正負のパターンが出る。moss の縁が全部 Fe XVIII に化ける。

In [ ]:
# 3 波長の WCS が同一であることを確認してから、同じ画素範囲で切る
for k in ("crpix1", "crpix2", "crval1", "crval2", "cdelt1", "crota2"):
    vals = {m.meta[k] for m in (a94, a171, a193)}
    assert len(vals) == 1, f"{k} が一致しない: {vals}"
print("3 波長の WCS は完全に一致 → 同じ画素範囲で切ってよい")

sub94 = a94.submap(bl, top_right=tr)
ny, nx = sub94.data.shape
px = a94.world_to_pixel(bl)
i0, j0 = int(round(px.y.value)), int(round(px.x.value))


def cut(m):
    return sunpy.map.Map(m.data[i0:i0 + ny, j0:j0 + nx], sub94.meta)


s94, s171, s193 = cut(a94), cut(a171), cut(a193)
s18 = sunpy.map.Map(fe18(s94.data, s171.data, s193.data), sub94.meta)
print("揃えた後:", s94.data.shape, s171.data.shape, s193.data.shape)

In [ ]:
panels = [("AIA 171 (moss, 0.9 MK)", s171, "sdoaia171"),
          ("AIA 193 (1.6 MK)", s193, "sdoaia193"),
          ("AIA 94 (raw)", s94, "sdoaia94"),
          ("AIA 94 -> Fe XVIII (7 MK)", s18, "inferno")]

fig = plt.figure(figsize=(16, 4.6))
for k, (title, m, cmap) in enumerate(panels):
    ax = fig.add_subplot(1, 4, k + 1, projection=m)
    d = np.sqrt(np.clip(m.data, 0, None))
    ax.imshow(d, origin="lower", cmap=cmap,
              vmin=np.nanpercentile(d, 1), vmax=np.nanpercentile(d, 99.7))
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    ax.set_xlabel("Solar X")
    ax.set_ylabel("Solar Y" if k == 0 else "")
    if k > 0:
        ax.coords[1].set_ticklabel_visible(False)
fig.tight_layout()
plt.show()

**見るべきこと**

- **171 Å**: 活動領域コアの中に**まだらの明るい模様** = moss。
  高温ループの足元が遷移層で光っている。
- **94 Å（生）**: 171 の moss がそのまま透けて見える。**これが汚染**。
- **Fe XVIII**: moss が消え、**滑らかで太いループ**だけが残る。
  これが 7 MK のプラズマ。

→ **Fe XVIII で明るく、171 の moss を含まない場所**が、
  論文の言う inter-moss 領域。次のモジュールで選ぶ。

## 3-5. 論文 Table 2 の最終行と答え合わせ

論文 Table 2 の最終行は EIS の輝線ではなく
**AIA 94 Å（Fe XVIII 分離後）の 7.20 ± 1.40 DN/s** で、
EIS と同じ inter-moss 箱で測った値。

**EIS とは完全に独立な測定**なので、箱の位置の検証に使える。
論文は箱の座標を書いていないが、Figure 2 の緑枠から実測できる
（`scripts/extract_paper_boxes.py`）。

In [ ]:
PAPER_BOX = dict(x0=-321.8, x1=-306.4, y0=202.6, y1=226.2)     # Fig.2 から実測
m18 = sunpy.map.Map(fe18(a94.data, a171.data, a193.data), a94.meta)

bl = SkyCoord(PAPER_BOX["x0"] * u.arcsec, PAPER_BOX["y0"] * u.arcsec,
              frame=m18.coordinate_frame)
tr = SkyCoord(PAPER_BOX["x1"] * u.arcsec, PAPER_BOX["y1"] * u.arcsec,
              frame=m18.coordinate_frame)
sub = m18.submap(bl, top_right=tr)
v = float(np.nanmean(sub.data))
print(f"論文の箱 X=[{PAPER_BOX['x0']}, {PAPER_BOX['x1']}] "
      f"Y=[{PAPER_BOX['y0']}, {PAPER_BOX['y1']}]  ({sub.data.shape[0]}x{sub.data.shape[1]} px)")
print(f"  Fe XVIII 平均 = {v:.3f} DN/s")
print(f"  論文 Table 2  = 7.200 ± 1.40 DN/s")
print(f"  ratio         = {v/7.20:.3f}")

**0.86 前後**になったはず。論文の誤差 ±19% の範囲に入っている。

**★ ただし、測り方で 10% 動く**。同じ箱でも

| 使ったデータ | Fe XVIII | 対論文 |
|---|---:|---:|
| synoptic 1024² (2.4″/px) をそのまま | 6.17 | 0.86 |
| フルディスク level-1 (4096², 0.6″/px) を EIS 格子に再投影 | 6.81 | 0.95 |

15″×23″ の箱は 2.4″/画素だと **6×10 画素しかない**。
構造のある場所を粗い格子で測ると、平均は端の扱いで簡単に 10% 動く。

**教訓**: 「小さい箱を粗い画像で測る」ときは、
**解像度そのものが系統誤差になる**。論文と比べる前に、
自分の測定がどちらの向きに偏りうるかを見積もっておく。

## 3-6. この経験式の適用限界（必ず知っておくこと）

1. **フレア中は使えない。** Fe XXIV 192.04 Å が 193 Å チャンネルに入るので、
   warm 成分を過大評価して Fe XVIII が負になる。
2. **非常に明るい moss でも破綻する。** 較正データに無い強度域。
3. **AIA の感度劣化補正を掛けてはいけない。**
   この経験式は劣化補正**なし**のデータ（2010 年 3 月）で導かれている。
   `aiapy.calibrate.degradation()` を掛けると係数と整合しなくなる。
   - ただし 2010 年の較正を 2011 年以降のデータに使うことの是非は
     それ自体が議論の種。**論文どおりの再現をするなら掛けない。**
4. より丁寧にやるなら、**Fe XVIII だけの応答関数**を作って DEM に入れる
   （公式の 94 Å 応答は低温線込みなので使えない）。
   → `scripts/aia94_fe18_response.py`。モジュール 7 で使う。

## 3-7. 演習

1. `f = 0.31` を 0.2 や 0.5 に変えて Fe XVIII マップがどう変わるか見る。
   moss の消え方が変わるはず。**どの f が「正しい」と言えるか？**
2. Fe XVIII が**負になる画素**を数えて、どこに分布するか描いてみる。
   （ヒント: 静穏領域と、非常に明るい moss）
3. 論文の経験式を**自分で導き直す**（発展）。
   静穏領域を含む広い領域で `x` と `I_94` の散布図を作り、3 次式を当てる。
   論文 Appendix の追体験になる。

## まとめ

- EIS の上限（~5 MK）より上を拘束するために **AIA 94 Å の Fe XVIII** を使う
- 94 Å は低温線に汚染されている → **171/193 から経験的に引く**
- **論文 Eq.(A1) の指数は誤植**。実データを通せば自分で確かめられる
- Fe XVIII で明るく 171 の moss が無い場所 = **inter-moss 領域**
- 小さい箱を粗い画像で測ると、**解像度が 10% の系統誤差**になる

次（モジュール 4）では、EIS と AIA の座標を合わせて
**実際に箱を選ぶ**。

# モジュール 4: EIS と AIA の座標合わせ、inter-moss 領域の選択

**所要時間 40 分**

**このノートで身につくこと**

1. **EIS のポインティングは信用できない**ことを自分の目で確かめ、
   相互相関で合わせる
2. AIA を EIS の画素格子に載せ替える（`reproject_to`）
3. **inter-moss 領域を選ぶ**。ここが解析で最も主観的な部分
4. 「どこを測るか」が結果を左右することを、次のモジュールへの伏線として持つ

前提: モジュール 1, 3。

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import sunpy.map
import eispac
from astropy.coordinates import SkyCoord

EIS_FILE = "data/eis/eis_20110702_030712.data.h5"
AIA_DIR = "data/sdo/synoptic"
AIA_BASE = "http://jsoc.stanford.edu/data/aia/synoptic/2011/07/02/H0300"


def ensure(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", os.path.basename(path))
    urllib.request.urlretrieve(url, path)
    return path


for ext in ("data", "head"):
    ensure(f"https://eis.nrl.navy.mil/level1/hdf5/2011/07/02/"
           f"eis_20110702_030712.{ext}.h5",
           f"data/eis/eis_20110702_030712.{ext}.h5")


def load_aia(wave):
    f = ensure(f"{AIA_BASE}/AIA20110702_0338_{wave:04d}.fits",
               f"{AIA_DIR}/AIA20110702_0338_{wave:04d}.fits")
    m = sunpy.map.Map(f)
    return sunpy.map.Map(m.data / m.meta["exptime"], m.meta)   # DN/s


a94, a171, a193 = load_aia(94), load_aia(171), load_aia(193)
print("AIA ok")

## 4-1. EIS のラスターを sunpy Map にする

座標を扱うには WCS 付きの Map がほしい。

**フィットは要らない**。相互相関に使うだけなので、
モジュール 1 のクイックルック（波長方向の積分）で十分。
全ラスターをフィットすると 3 分かかるが、これなら 1 秒。

In [ ]:
def eis_raster_map(datafile, wvl):
    """EIS のスペクトルウィンドウを波長方向に積んで sunpy Map にする。"""
    c = eispac.read_cube(datafile, wvl)
    d = np.where(np.asarray(c.mask, dtype=bool), np.nan, c.data)   # 欠損を除く
    img = np.nanmean(d, axis=2) * d.shape[2]

    h, p = c.meta["index"], c.meta["pointing"]
    ref = SkyCoord(p["xcen"] * u.arcsec, p["ycen"] * u.arcsec,
                   obstime=h["date_obs"], observer="earth", frame="helioprojective")
    hdr = sunpy.map.make_fitswcs_header(
        img, ref, scale=[p["x_scale"], p["y_scale"]] * u.arcsec / u.pix,
        instrument="EIS", wavelength=wvl * u.angstrom)
    hdr["measrmnt"] = "intensity"          # eispac の EISMap が要求するキー
    return sunpy.map.Map(img, hdr)


m_eis = eis_raster_map(EIS_FILE, 195.119)
print("EIS Fe XII map:", m_eis.data.shape)
print(f"  X = {m_eis.bottom_left_coord.Tx.value:.1f} .. "
      f"{m_eis.top_right_coord.Tx.value:.1f}\"")
print(f"  Y = {m_eis.bottom_left_coord.Ty.value:.1f} .. "
      f"{m_eis.top_right_coord.Ty.value:.1f}\"")
print(f"  画素 = {m_eis.scale[0]:.2f} x {m_eis.scale[1]:.2f}")

## 4-2. AIA を EIS の格子に載せ替える

`reproject_to` は WCS を見て座標変換してくれる。
EIS の格子は x 2″ × y 1″ という**変な格子**だが、気にせず載せられる。

In [ ]:
r193, r171, r94 = (m.reproject_to(m_eis.wcs) for m in (a193, a171, a94))
print("AIA on EIS grid:", r193.data.shape)

A = [-7.31e-2, 9.75e-1, 9.90e-2, -2.84e-3]


def fe18(i94, i171, i193, f=0.31):
    x = np.clip((f * i171 + (1 - f) * i193) / 116.54, 0.0, 30.0)
    return i94 - 0.39 * sum(a * x**i for i, a in enumerate(A))


fe = fe18(r94.data, r171.data, r193.data)

## 4-3. ★ ずれているのを見る

EIS Fe XII 195.119（1.6 MK）と AIA 193（1.6 MK）は**ほぼ同じ温度**なので、
形態がよく似ているはず。並べて見る。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 9))
for ax, (d, t) in zip(axes, [(m_eis.data, "EIS Fe XII 195.1"),
                             (r193.data, "AIA 193 (on EIS grid)")]):
    v = np.sqrt(np.clip(d, 0, None))
    lo, hi = np.nanpercentile(v, [1, 99.5])
    ax.imshow(v, origin="lower", aspect="auto", cmap="inferno", vmin=lo, vmax=hi)
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("EIS x [pix]")
axes[0].set_ylabel("EIS y [pix]")
fig.tight_layout()
plt.show()

よく似ているが、**同じ場所に重なっていない**。

**なぜずれるか**

- EIS のポインティングには **数″〜十数″の系統誤差**がある
  （熱変形、姿勢基準の違い、軌道中の指向ドリフト）
- ヘッダの `xcen`/`ycen` を鵜呑みにすると、選んだ箱が別の場所を指す
- 論文は箱を「AIA Fe XVIII で明るく AIA 171 の moss が無い場所」として
  選んでいるので、**この座標合わせの精度がそのまま強度の精度になる**

## 4-4. 相互相関でずれを測る

整数画素精度で十分（EIS の画素は 1″ × 2″）。

**★ 相関係数の取り方に注意**: 重なり領域**ごとに**正規化した Pearson 相関を使う。
配列全体で一度だけ正規化すると、重なりが小さいシフトほど見かけの相関が
上がってしまい、ずれを過大評価する（実際にこれで一度間違えた）。

In [ ]:
def cross_correlate_shift(ref, img, max_shift=25):
    """img を ref に合わせるための (dy, dx) を返す。"""
    def prep(a):
        a = np.array(a, float)
        a[~np.isfinite(a)] = np.nanmedian(a)
        return np.sqrt(np.clip(a, 0, None))     # 明るいコアだけで決まらないように

    r, i = prep(ref), prep(img)
    ny, nx = r.shape
    best, bdy, bdx = -np.inf, 0, 0
    for dy in range(-max_shift, max_shift + 1):
        for dx in range(-max_shift, max_shift + 1):
            rs = r[max(0, dy):ny + min(0, dy), max(0, dx):nx + min(0, dx)]
            is_ = i[max(0, -dy):ny + min(0, -dy), max(0, -dx):nx + min(0, -dx)]
            if rs.size < 0.5 * r.size:
                continue
            rc, ic = rs - rs.mean(), is_ - is_.mean()
            den = np.sqrt((rc**2).sum() * (ic**2).sum())
            c = float((rc * ic).sum() / den) if den > 0 else -np.inf
            if c > best:
                best, bdy, bdx = c, dy, dx
    return bdy, bdx, best


dy, dx, cc = cross_correlate_shift(m_eis.data, r193.data)
print(f"ずれ: dy = {dy} pix (= {dy*1.0:.0f}\"),  dx = {dx} pix "
      f"(= {dx*2.0:.0f}\")   相関 {cc:.3f}")

In [ ]:
def shift(a, dy, dx):
    out = np.full_like(np.asarray(a, float), np.nan)
    ny, nx = a.shape
    out[max(0, dy):ny + min(0, dy), max(0, dx):nx + min(0, dx)] = \
        a[max(0, -dy):ny + min(0, -dy), max(0, -dx):nx + min(0, -dx)]
    return out


a193s, a171s, fes = (shift(v, dy, dx) for v in (r193.data, r171.data, fe))

fig, axes = plt.subplots(1, 4, figsize=(13, 9))
for ax, (d, t) in zip(axes, [(m_eis.data, "EIS Fe XII 195.1"),
                             (a193s, "AIA 193 (shifted)"),
                             (a171s, "AIA 171 = moss"),
                             (fes, "AIA Fe XVIII = 7 MK")]):
    v = np.sqrt(np.clip(d, 0, None))
    lo, hi = np.nanpercentile(v, [1, 99.5])
    ax.imshow(v, origin="lower", aspect="auto", cmap="inferno", vmin=lo, vmax=hi)
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("EIS x [pix]")
axes[0].set_ylabel("EIS y [pix]")
fig.suptitle(f"co-aligned  (dy={dy}, dx={dx} pix)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 4-5. inter-moss 領域を選ぶ

論文の言い方:

> the inter-moss region, that is, the region between the loop footpoints where
> we are measuring the properties near the loop apex

**なぜ moss を避けるのか**

- moss = 高温ループの**足元**が遷移層で光っているもの（171 Å で明るい）
- moss を含むと、視線上に「足元の 1 MK」と「ループ上部の 4 MK」が混ざる
- 足元は熱伝導・彩層蒸発が絡む複雑な物理。**ループ上部だけを見たい**

**数値化**: `score = median(Fe XVIII) / median(AIA 171)`
（高温で明るく、moss が暗いほど大きい）

**★ ただし最後は図を見て人間が決める。** 自動化しきらないのが正しい。
物理的な判断だからである。

In [ ]:
def scan(fe, a171, ny, nx, step=4, min_fe=3.0):
    H, W = fe.shape
    out = []
    for y0 in range(0, H - ny, step):
        for x0 in range(0, W - nx, step):
            f, m = fe[y0:y0+ny, x0:x0+nx], a171[y0:y0+ny, x0:x0+nx]
            if not (np.isfinite(f).all() and np.isfinite(m).all()):
                continue
            fmed, mmed = np.median(f), np.median(m)
            if fmed < min_fe:            # Fe XVIII が暗い場所は候補にしない
                continue
            out.append((fmed / mmed, fmed, mmed, y0, y0+ny, x0, x0+nx))
    return sorted(out, reverse=True)


print(f"{'score':>8} {'FeXVIII':>9} {'AIA171':>8}   box")
cands = scan(fes, a171s, 30, 8)
for c in cands[:8]:
    s, f, m, y0, y1, x0, x1 = c
    print(f"{s:8.4f} {f:9.2f} {m:8.0f}   y=[{y0}:{y1}] x=[{x0}:{x1}]")

**★ 1 位に出てきた `y=[244:274] x=[32:40]` は、我々が採用した箱そのもの。**

この箱は準備段階で「論文 Table 2 に最もよく合う場所」として
22 輝線の突き合わせから選んだものだが、
**論文が書いている選択基準（Fe XVIII で明るく moss が無い）だけを
機械的に適用しても同じ場所に来る**。

選択基準が言葉どおりに再現できている、という確認になる。
逆に言えば、**この一致が無ければ「たまたま合う箱を探した」ことになる**。
答えを知っている問題では、ここを分けて考えるのが大事。

### 採用する箱

上位候補と、**論文が使った箱**（Figure 2 の緑枠から実測したもの、
`scripts/extract_paper_boxes.py`）を重ねて見る。

論文は箱の座標を書いていないので、**図から読むしかない**。
論文の箱は 15.4″ × 23.5″。他の論文（Winebarger et al. 2011）も
5″ × 25″ と**縦長の細い箱**を使っており、それがこのグループの流儀。

In [ ]:
BOX = dict(y0=244, y1=274, x0=32, x1=40)       # 準備段階で採用した箱

fig, axes = plt.subplots(1, 3, figsize=(11, 9))
for ax, (d, t) in zip(axes, [(fes, "AIA Fe XVIII"),
                             (a171s, "AIA 171 (moss)"),
                             (m_eis.data, "EIS Fe XII 195.1")]):
    v = np.sqrt(np.clip(d, 0, None))
    lo, hi = np.nanpercentile(v, [1, 99.5])
    ax.imshow(v, origin="lower", aspect="auto", cmap="inferno", vmin=lo, vmax=hi)
    for c in cands[:8]:                        # 候補（水色）
        _, _, _, y0, y1, x0, x1 = c
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0],
                color="cyan", lw=0.7, alpha=0.5)
    b = BOX                                    # 採用（白）
    ax.plot([b["x0"], b["x1"], b["x1"], b["x0"], b["x0"]],
            [b["y0"], b["y0"], b["y1"], b["y1"], b["y0"]], color="white", lw=2)
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("EIS x [pix]")
axes[0].set_ylabel("EIS y [pix]")
fig.suptitle("inter-moss candidates (cyan) and adopted box (white)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
b = BOX
print(f"採用箱 y=[{b['y0']}:{b['y1']}] x=[{b['x0']}:{b['x1']}]"
      f"  = {b['y1']-b['y0']}\" x {(b['x1']-b['x0'])*2}\"")
v18 = float(np.nanmean(fes[b["y0"]:b["y1"], b["x0"]:b["x1"]]))
print(f"  AIA Fe XVIII 平均 = {v18:6.2f} DN/s"
      f"   （論文 Table 2 = 7.20 ± 1.40 → 比 {v18/7.20:.2f}）")
print(f"  AIA 171     平均 = {np.nanmean(a171s[b['y0']:b['y1'], b['x0']:b['x1']]):6.0f} DN/s"
      f"   （視野の中央値 {np.nanmedian(a171s):.0f}）")

os.makedirs("data/cache", exist_ok=True)
np.savez("data/cache/aia_on_eis_grid.npz", aia171=a171s, aia193=a193s, fe18=fes,
         eis_fe12=m_eis.data, dy=dy, dx=dx)
print("\nwrote data/cache/aia_on_eis_grid.npz  （モジュール 5, 7 で使う）")

## 4-6. ★ ここが解析で一番主観的なところ

論文は**目で見て手で箱を選んでおり、座標を書いていない**。
我々が Figure から復元した箱と、指標で選んだ箱は近いが同じではない。

準備段階で 230 通りの箱を総当たりした結果:

| 箱の選び方 | 論文 Table 2 との median 比 |
|---|---|
| 適当に明るいところ | 0.3 〜 1.2 まで散らばる |
| inter-moss の条件を満たす箱 | 0.83 〜 0.95 |

**箱を動かすだけで結果は 2 割動く。** これは論文の誤差 ±22% と同程度。

→ **「どこを測るかを決めるのが解析の本体」**である。
  装置やコードの議論より、まずここを疑う。

次のモジュールでは、この箱で出した 22 輝線の強度を論文と突き合わせ、
**箱の選び方が正しいかを数値で診断する**方法を学ぶ。

## 4-7. 演習

1. 相互相関の相手を **AIA 171** に変えるとどうなるか。
   Fe XII (1.6 MK) と 171 (0.9 MK) では形態が違うので、
   ずれの推定が悪化するはず。**温度の近い組を選ぶ**理由を確かめる。
2. `max_shift` を 5 にすると答えが変わるか。境界に張り付いていないか確認する。
3. 上位候補の箱をいくつか `BOX` に入れて、
   モジュール 2 の 22 輝線フィットを回す。ratio がどう動くか。
4. `min_fe`（Fe XVIII の下限）を 1.0 に下げると、どんな場所が候補に入ってくるか。

## まとめ

- **EIS のポインティングはずれている**。AIA との相互相関で合わせる
- 相手は**温度の近い線**（EIS Fe XII ↔ AIA 193）を選ぶ
- inter-moss = **Fe XVIII で明るく、171 の moss が無い**場所。ループ上部を見るため
- **箱の選び方だけで結果は 2 割動く**。ここが解析で最も主観的

次（モジュール 5）は講習会の山場、**論文 Table 2 との答え合わせ**。

# モジュール 5: 論文 Table 2 と答え合わせする ★講習会の山場

**所要時間 30 分**

**このノートで身につくこと**

1. 自分が出した 22 輝線の強度を、論文 Table 2 と**1 行ずつ**突き合わせる
2. **「箱の選び方」を数値で診断する**（ratio の温度依存の傾き）
3. 合わない線について、**原因を切り分ける手順**を身につける
4. ★ **独立な 2 つ目の測定を持ってくる**のが最強の切り分けだと理解する

前提: モジュール 2（22 輝線のフィット）、モジュール 4（箱の選択）。

---

**なぜこの活動領域なのか**: 論文 Table 2 は region 7（2011-07-02 03:07:12,
NOAA 1243）の観測強度を 23 行そのまま載せている。
**論文に数値表が載っている唯一の活動領域**なので、
「自分の解析が合っているか」が客観的に確かめられる。

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import eispac

EIS_FILE = "data/eis/eis_20110702_030712.data.h5"
BOX = dict(y0=244, y1=274, x0=32, x1=40)          # モジュール 4 で選んだ箱


def ensure(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(url, path)
    return path


for ext in ("data", "head"):
    ensure(f"https://eis.nrl.navy.mil/level1/hdf5/2011/07/02/"
           f"eis_20110702_030712.{ext}.h5",
           f"data/eis/eis_20110702_030712.{ext}.h5")

import sys
sys.path.insert(0, "scripts")                      # 教材リポジトリの中
from lines_warren2012 import LINES, pick_component
from fit_box_spectra import average_spectrum       # モジュール 2 と同じ中身
print(f"論文 Table 2 の輝線数: {len(LINES)}")

## 5-1. 22 輝線をフィットして比べる

In [ ]:
# おおまかな形成温度（log T）。ratio の温度依存を見るために使う
LOGT = {"Si VII": 5.8, "Fe IX": 5.9, "Fe X": 6.05, "Fe XI": 6.15, "S X": 6.15,
        "Si X": 6.15, "Fe XII": 6.2, "Fe XIII": 6.25, "Fe XIV": 6.3,
        "Fe XV": 6.35, "S XIII": 6.4, "Fe XVI": 6.45, "Ar XIV": 6.5,
        "Ca XIV": 6.55, "Ca XV": 6.65, "Ca XVI": 6.7, "Ca XVII": 6.75}


def fit_box(box):
    rows = []
    for ion, wvl, tname, i_paper, sig_paper in LINES:
        w, I, s, npix = average_spectrum(EIS_FILE, wvl, **box)
        t = eispac.read_template(eispac.data.get_fit_template_filepath(tname))
        comp, ids = pick_component(t, wvl)
        f = eispac.fit_spectra(I, t, wave=w, errs=s, ncpu=1, ignore_warnings=True)
        i_fit = float(np.atleast_1d(f.fit["int"][..., comp]).ravel()[0])
        rows.append(dict(ion=ion, wvl=wvl, i_fit=i_fit, i_paper=i_paper,
                         sig_paper=sig_paper, ratio=i_fit / i_paper,
                         logt=LOGT[ion]))
    return rows


rows = fit_box(BOX)

print(f"{'line':<16}{'logT':>5}{'I_fit':>10}{'I_paper':>10}{'±':>8}{'ratio':>7}")
for r in sorted(rows, key=lambda r: r["logt"]):
    flag = "  ✓" if abs(r["ratio"] - 1) <= 0.15 else ""
    name = "{} {:.3f}".format(r["ion"], r["wvl"])
    print(f"{name:<16}{r['logt']:5.2f}{r['i_fit']:10.2f}{r['i_paper']:10.2f}"
          f"{r['sig_paper']:8.2f}{r['ratio']:7.2f}{flag}")

In [ ]:
use = [r for r in rows if r["ion"] != "Ca XVII"]     # Ca XVII はブレンド（後述）
ratios = np.array([r["ratio"] for r in use])
logt = np.array([r["logt"] for r in use])
n15 = int((np.abs(ratios - 1) <= 0.15).sum())

print(f"Ca XVII を除く {len(use)} 本について")
print(f"  median ratio  = {np.median(ratios):.2f}")
print(f"  ばらつき      = {np.std(np.log10(ratios)):.2f} dex")
print(f"  論文の 15% 以内 = {n15}/{len(use)} 本")
print(f"  論文自身の誤差  = ±22%")

**まず全体を見る。median が 0.89 = 論文より 11% 低い。**

これは論文の誤差 ±22% の中に収まっている。
「合っている」と言ってよい水準だが、**なぜ 11% 低いのか**は次節で切り分ける。

## 5-2. ★ 「箱の選び方」を数値で診断する

ratio を**形成温度に対して**並べるのが決め手になる。

| ratio の温度依存 | 意味 |
|---|---|
| 傾き ≈ 0（全体に一定倍率） | 論文と**同じ温度組成**の場所を見ている。ずれは較正の問題 |
| 傾き < 0（冷たい線が明るく熱い線が暗い） | **暖かいループ寄り**を選んでいる |
| 傾き > 0（熱い線が明るい） | **高温コア寄り**を選んでいる |

装置の較正は波長の関数であって温度の関数ではないから、
**温度に沿ったパターンが出たら、それは場所の違い**である。

In [ ]:
slope = np.polyfit(logt, np.log10(ratios), 1)[0]
print(f"log(ratio) の logT に対する傾き = {slope:+.2f}")

plt.figure(figsize=(7, 4.5))
for r in use:
    plt.plot(r["logt"], r["ratio"], "o", color="C0")
    if abs(r["ratio"] - 1) > 0.25:
        plt.annotate(f"{r['ion']} {r['wvl']:.1f}", (r["logt"], r["ratio"]),
                     fontsize=7, xytext=(3, 3), textcoords="offset points")
xx = np.linspace(logt.min(), logt.max(), 10)
plt.plot(xx, 10**np.polyval(np.polyfit(logt, np.log10(ratios), 1), xx),
         "-", color="C3", label=f"slope = {slope:+.2f}")
plt.axhline(1.0, color="k", lw=1)
plt.axhspan(0.85, 1.15, color="0.85", zorder=0, label="within 15%")
plt.yscale("log")
plt.xlabel("log T [K]")
plt.ylabel("I(ours) / I(Warren+2012)")
plt.title("ratio vs formation temperature")
plt.legend()
plt.tight_layout()
plt.show()

### 箱を変えて傾きを比べる（この演習が一番教育的）

**受講者に何通りか箱を選ばせて、この傾きを比べさせる**のが良い。

In [ ]:
BOXES = {
    "採用箱 (inter-moss)":  dict(y0=244, y1=274, x0=32, x1=40),
    "論文の箱サイズに近い":  dict(y0=246, y1=269, x0=30, x1=38),
    "適当に明るいところ":    dict(y0=200, y1=230, x0=20, x1=28),
}
for label, box in BOXES.items():
    rr = fit_box(box)
    u = [r for r in rr if r["ion"] != "Ca XVII"]
    ra = np.array([r["ratio"] for r in u])
    lt = np.array([r["logt"] for r in u])
    sl = np.polyfit(lt, np.log10(ra), 1)[0]
    print(f"{label:<22} median={np.median(ra):5.2f}  傾き={sl:+5.2f}  "
          f"15%以内={int((np.abs(ra-1) <= 0.15).sum())}/{len(u)}  "
          f"ばらつき={np.std(np.log10(ra)):.2f} dex")

**★ ここは必ず立ち止まって読むこと。**

「適当に明るいところ」の箱は、**median が 0.96 と一番 1 に近い**。
median だけ見ていたら「この箱が一番良い」と結論してしまう。

ところが同じ箱は

- **傾き −0.34**（冷たい線が明るく熱い線が暗い = 暖かいループを見ている）
- **15% 以内が 5/21 本しかない**（採用箱は 13/21）
- **ばらつき 0.26 dex**（採用箱の 2.6 倍）

つまり **個々の線は全然合っていないのに、上下のずれが打ち消し合って
median だけがきれいに見えている**。

→ **要約統計 1 つで判断しない。** 温度に沿って並べる、ばらつきを見る、
  本数を数える。この 3 つを揃えて初めて「合っている」と言える。

## 5-3. 合わない線を 1 つずつ議論する（ここが一番勉強になる）

### (a) Ca XVII 192.858 が 5 倍 → **原因がはっきりしている**

eispac 同梱の `ca_17_192_858.1c` は 192.700–193.200 Å を
**単一ガウシアンで塗るだけ**で、Fe XI 192.813 と O V の複合線を分離しない。
論文は Ko et al. (2009) の方法で分離している。

→ **モジュール 8** で自作テンプレートを作ると **5.08 → 0.75** になる。
  （SSW/IDL で同じことをすると 0.77。Python だけで再現できる）

### (b) Fe XIII 202.044 / 203.826 → **論文でも外れている**

後で DEM を解くと、この 2 本だけ I_obs/I_DEM が 1.3–2.8 になる。
**論文でも 1.80 / 1.82**、Warren et al. (2011) でも 1.87 / 1.90。
→ **原子データ側の既知の問題**であって、我々の解析の問題ではない。
  密度診断ペアであり、モジュール 6 で「実装間の差が最大の線」として再登場する。

### (c) Si VII 275.368 が 0.40、Fe XVI 262.984 が 0.62 → **未解決**

ここが正直に扱うべきところ。**容疑を 1 つずつ潰した記録**を次に示す。

### ★ 「Aで説明できる」と言ったら、Aの予測を検証する

**仮説 1: inter-moss 領域の自然なばらつき**

他論文の inter-moss 領域と、Fe XII 195.119 に対する比で並べると:

| | SiVII/FeXII | FeXVI/FeXII | SXIII/FeXII |
|---|---:|---:|---:|
| Tripathi+2011 inter-moss A | 0.0477 | 0.4381 | 0.4704 |
| Tripathi+2011 inter-moss B | 0.0230 | 0.2372 | 0.2916 |
| Tripathi+2011 inter-moss C | 0.0848 | 0.4763 | 0.5267 |
| Warren+2011 | 0.0319 | 0.7846 | 0.5793 |
| Warren+2012 region 7（論文） | 0.0583 | 0.5498 | 0.4029 |
| **我々** | **0.0240** | **0.3620** | **0.3078** |

実在の inter-moss 領域どうしで 3.7 倍ばらついている。
「だから正常」と言いたくなる。**しかしこれは誤り。**
Tripathi+2011 は**別の活動領域**であり、
「このラスターの中に論文と同じ組み合わせが実現できる」ことを保証しない。

**仮説が正しいなら成り立つはずの予測**:
「自然なばらつきなら、このラスター内に論文と同じ比の箱があるはず」

→ 230 箱を総当たりして検証した結果（`scripts/scan_perline.py`）:

- **「22 輝線の 15% 以内が 12 本以上」かつ「Si VII 比 ≥ 0.8」を満たす箱は 0 個**
- Si VII を合わせにいくと Si X 1.85、Fe XIV 1.67、Ar XIV 0.28、
  Ca XVI 0.15 と**高温側が壊滅する**

→ **予測は反証された。仮説 1 は棄却。**

### ★★ 決め手は AIA（EIS と完全に独立な測定）

論文 Table 2 の最終行は **AIA 94 Å の Fe XVIII = 7.20 DN/s**。
分光器 (EIS) とは別の装置・別のデータ経路・別の単位。

| 箱 | AIA Fe XVIII | 対論文 | SiVII 比 |
|---|---:|---:|---:|
| Si VII が合う箱 | 1.16 | **0.16** | 1.13 |
| 我々の採用箱 | 6.64 | **0.92** | 0.36 |
| 論文の箱（Fig.2 から実測） | 6.81 | **0.95** | 0.43 |

**Si VII が合う箱は AIA Fe XVIII が論文の 6 分の 1**。
そこは moss であって inter-moss ではない、と **EIS とは独立に**言える。

逆に AIA が論文と合う箱では、**Si VII は必ず 2.6 倍低い**。

→ **論文 Table 2 の Si VII 値は、同じ Table 2 の AIA Fe XVIII 値と整合しない。**
  ここまで絞れれば、著者に問い合わせる価値がある。

**潰した容疑（10 個）**: フィッターの実装 / eispac 固有の問題 / 箱の位置 /
打ち上げ後較正 2 種（Del Zanna 2013, Warren+2014）/ 実効面積のバージョン /
despike / 欠損値処理 / フィットの順番 / 未モデル化のブレンド /
「箱で説明できる」説。

**★ 「長波長側の感度劣化」も成立しない。** 同じ長波長チャンネルの
S X (0.94)、Si X (1.09)、Fe XIV (0.92/0.95)、Fe XV (0.87) は合っている。
**隣り合う波長**の Si X 258.375 と Fe XVI 262.984 が 1.09 と 0.62 に分かれるので、
波長の滑らかな関数である較正では説明できない。

## 5-4. ★★★ 決定的な事実: 独立な 2 台が揃って低い

| | 我々 | 論文 |
|---|---:|---:|
| EIS 22 輝線の median | **0.89** | 1.0 |
| AIA 94 Å Fe XVIII | **0.82 – 0.92**（測り方による） | 1.0 |

**分光器（EIS, erg 単位, NRL の level-1）と撮像（AIA, DN 単位, JSOC）という
完全に独立な 2 台が、揃って 1 割低い。**

→ 装置でも処理でもなく、**論文がわずかに明るい場所を測っている**と結論できる。
  差は論文自身の誤差 ±22% の中。

（AIA の値に幅があるのは、synoptic 1024²(2.4″/px) で測るか
  フルディスク level-1 (4096²) を EIS 格子に落として測るかの違い。
  15″×23″ の箱を粗い格子で測ると 10% 動く。モジュール 3 参照。）

## 5-5. この章の教訓

1. **一致したことより、合わない理由を説明できることが実力。**
   22 本中 13 本が 15% 以内、というのは結果の半分でしかない。
2. **「Aで説明できる」と言ったら、Aが正しいなら成り立つ予測を立てて検証する。**
   今回は「このラスター内に論文と同じ箱があるはず」という予測が反証された。
3. **独立な 2 つ目の測定を持ってくるのが最強の切り分け手段。**
   今回は AIA の Fe XVIII が、EIS とは全く別ルートの検証になった。
4. **未解決を未解決のまま正確に記述する。**
   「Si VII が合わない」ではなく
   「論文 Table 2 の Si VII 値は、同じ Table 2 の AIA 値と整合しない」
   まで絞る。ここまで来て初めて次の一手が決まる。

## 5-6. 演習

1. `BOXES` に自分で箱を足して、median と傾きの組を集める。
   **median が 1 に近くても傾きが大きい箱**を見つけられるか？
   見つかったら、それは何を意味するか。
2. Fe XII 195.119 で規格化した比（`I / I_FeXII195`）で論文と比べ直す。
   絶対較正の効果が落ちるので、**場所の違いだけ**が見えるはず。
3. Ca XVII を除かずに median を計算するとどうなるか。
   **1 本の外れ値が要約統計をどれだけ動かすか**を体感する。
4. 論文の誤差 ±22% を図に描き込んで、
   「何本が誤差の範囲内か」を数え直す。

## まとめ

- 22 輝線中 **13 本が論文の 15% 以内**、median 0.89、ばらつき 0.10 dex
- **ratio の温度依存の傾き**で箱の選び方を診断できる
- Ca XVII = ブレンド（解ける）、Fe XIII = 原子データ（論文も同じ）、
  Si VII / Fe XVI = **未解決**
- **独立な 2 台（EIS と AIA）が揃って 1 割低い** → 場所の違い

ここまでが「半日コース」の到達点。
以降は寄与関数（モジュール 6）と DEM（モジュール 7）に進む。

# モジュール 6: 寄与関数 G(T) と EM loci

**所要時間 40 分**

**このノートで身につくこと**

1. 寄与関数 G(T) が何でできているか（組成・電離平衡・励起）を理解する
2. **輝線ごとに効く温度が違う**ことを数値で見る（これが DEM の武器）
3. ★ **1/(4π) の罠**を知る。実装によって G(T) が 12.6 倍ずれる
4. **EM loci** を描いて、逆問題を解く前に答えの見当をつける

前提: モジュール 2（22 輝線の強度）。

---

**Colab での方針**: CHIANTI のデータベースは 257 MB あり、
セッションが切れると消える。そこで **事前計算した G(T) をリポジトリに同梱**し、
既定ではそれを読む。自分で計算する経路（fiasco）も 6-4 に用意した。

In [ ]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt

GOFNT = "work/gofnt_chianti901.txt"          # 0.1 dex 刻み（MCMC 用）
GOFNT_FINE = "work/gofnt_chianti901_005.txt"  # 0.05 dex 刻み（demregpy 用）
print(open(GOFNT).read().split("# nT")[0])   # ヘッダ = 何を仮定して作ったか

## 6-1. G(T) は 3 つの因子でできている

コロナは光学的に薄いので、視線積分は**単なる足し算**:

$$ I_\lambda = \frac{1}{4\pi}\int G_\lambda(T)\, n_e n_H\, ds $$

$$ G_\lambda(T) = \underbrace{A(Z)}_{組成}
   \times \underbrace{f_{\rm ion}(T)}_{電離平衡}
   \times \frac{n_H}{n_e}
   \times \underbrace{\frac{hc}{\lambda}\frac{n_j}{n_{\rm ion}}A_{ji}\frac{1}{n_e}}_{励起（準位占有数）} $$

| 因子 | 何で決まるか | 不確かさ |
|---|---|---|
| 組成 $A(Z)$ | コロナ組成 (Feldman 1992) か光球組成か | **FIP 効果で 3-4 倍** |
| 電離平衡 $f_{\rm ion}(T)$ | 電離・再結合レート | 数十 % |
| 励起 | 衝突励起断面積、準位占有数 | 数 %〜数十 % |

**G(T) が温度の狭い関数になるのは、主に電離平衡が狭いから。**
各イオンは log T で 0.2–0.3 dex の幅でしか存在しない
（温度が低ければまだ電離しておらず、高ければさらに電離してしまう）。

**★ これは「電離平衡が成り立っている」という仮定の上の話。**
急激な加熱・冷却では電離が追いつかない（非平衡電離）。論文もこの仮定に立つ。

## 6-2. 事前計算した G(T) を読んで描く

In [ ]:
def read_gofnt(path):
    """09_gofnt.pro / gofnt_fiasco.py が書く形式を読む。"""
    lines = open(path).readlines()
    i = next(k for k, l in enumerate(lines) if l.startswith("# nT nline"))
    nT, nline = (int(v) for v in lines[i + 1].split())
    k = i + 2

    def skip(tag):
        nonlocal k
        while not lines[k].startswith(tag):
            k += 1
        k += 1

    def take(n):
        nonlocal k
        out = []
        while len(out) < n:
            out += [float(x) for x in lines[k].split()]
            k += 1
        return np.array(out[:n])

    skip("# logT")
    logT = take(nT)
    skip("# ion")
    names, wvl = [], []
    for _ in range(nline):
        p = lines[k].split()
        names.append(" ".join(p[:-2]))
        wvl.append(float(p[-2]))
        k += 1
    skip("# G(T)")
    G = np.array([take(nT) for _ in range(nline)])
    return logT, names, np.array(wvl), G


logT, names, wvl, G = read_gofnt(GOFNT)
print(f"{len(names)} 輝線 x {len(logT)} 温度点  "
      f"(logT {logT[0]:.1f}-{logT[-1]:.1f}, {logT[1]-logT[0]:.2f} dex 刻み)")

print(f"\n{'line':<18}{'G のピーク':>12}{'ピーク温度 logT':>16}{'T [MK]':>9}")
for k in np.argsort([logT[np.argmax(g)] for g in G]):
    j = int(np.argmax(G[k]))
    print(f"{names[k]+' '+f'{wvl[k]:.3f}':<18}{G[k][j]:12.3e}"
          f"{logT[j]:16.2f}{10**logT[j]/1e6:9.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cmap = plt.get_cmap("turbo")
tpk = np.array([logT[int(np.argmax(g))] for g in G])
norm = plt.Normalize(tpk.min(), tpk.max())
for k in range(len(names)):
    ax.plot(logT, G[k] / G[k].max(), color=cmap(norm(tpk[k])), lw=1.2)
ax.set_xlim(5.4, 7.2)
ax.set_xlabel("log T [K]")
ax.set_ylabel("G(T) / max")
ax.set_title("contribution functions, normalized (color = peak temperature)")
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
             label="log T at peak")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**これが DEM 解析の武器そのもの。**
22 本の輝線が log T = 5.8 から 6.9 まで**少しずつずれた温度**に効いている。
各輝線の強度は「その温度あたりのプラズマ量」を測っているので、
22 個の測定値から温度分布を復元できる——ように見える。

**★ しかし曲線の幅に注目**。どれも 0.3–0.5 dex の幅があり、大きく重なっている。
だから「22 個の独立な情報」にはならない。これがモジュール 7 の主題。

## 6-3. ★★ 1/(4π) の罠 —— 静かに 12.6 倍ずれる

準備段階で、**同じ CHIANTI 9.0.1 の ASCII ファイル**を 3 つの実装に
読ませて G(T) を比べた（`scripts/idl/09_gofnt.pro`,
`scripts/gofnt_fiasco.py`, `scripts/gofnt_chiantipy.py`）。
比べているのは「DB のバージョン差」ではなく **実装の差だけ**。

| 実装 | 単位の約束 | CHIANTI IDL との比 |
|---|---|---|
| CHIANTI IDL `emiss_calc` | hc/λ × N_j × A_ji（**4π で割らない**） | 1.000 |
| fiasco `contribution_function` | 同上（4π で割らない） | 0.972 – 1.035 |
| **ChiantiPy `ion.emiss()`** | **sr⁻¹（4π で割ってある）** | **0.077 – 0.082** |

**1/(4π) = 0.0796。** 実測の median は 0.080 でぴったり一致する。
4π を掛け戻すと median 1.000（0.968–1.030）で他の 2 者と揃う。

**なぜこれが恐ろしいか**

- **エラーは一切出ない。** 静かに 12.6 倍ずれた DEM が出てくる
- **全輝線が一律にずれる**ので、線ごとの ratio を見ている限り絶対に気づけない
- DEM の絶対値は EM = n_e² L に直結するので、
  **ループの長さや密度の議論が丸ごと狂う**

気づく方法はただ 1 つ、**オーダーが物理的に妥当かを見る**こと。

In [ ]:
print("オーダーの検算（覚えておく数字）")
print(f"  Fe XII 195.119 の G ピーク = "
      f"{G[int(np.argmin(np.abs(wvl-195.119)))].max():.2e} erg cm^3 s^-1")
print("    → 10^-23 のオーダーなら正しい。10^-24 なら 4π で割られている")
print("  活動領域の EM (視線積分)   = 10^27 - 10^29 cm^-5")
print("  コロナの電子密度 n_e       = 10^9 cm^-3 （活動領域コア）")
print("  → EM = n_e^2 L より L = EM / n_e^2 = 1e28/1e18 = 1e10 cm = 100 Mm")
print("    ループの長さとして妥当。ここが 4 桁ずれたら単位を疑う")

**★ 実話**: 準備段階で MCMC_DEM の初期 DEM が 7.4e-7 cm⁻⁵ になった。
`emiss_calc` を n_e で割り忘れていたため（1e9 のずれ）。
**このオーダー感覚があったから即座に気づけた。**

もう 1 つの実測結果（教材として一級品）:
4π を補正すると **2 つの Python 実装は 4 桁一致**（0.999–1.000）し、
**IDL だけが最大 3.5% 違う**。差が大きいのは
**Fe XIII 202.044 / 203.826、Fe XIV 264.787、Si X 258.375** —— すべて**密度敏感線**。
差がほぼ無いのは Fe IX、Ca XIV、Ca XVI、Fe XV —— **基底準位からの共鳴線**。

→ 差の正体は**準安定準位の占有数の扱い**（陽子励起、励起準位への電離・再結合）。
→ **Fe XIII が DEM で唯一大きく外れる**（我々 1.3–2.8、論文 1.80/1.90）ことの
  独立な説明になっている。コードが違うだけで 3–12% 変わる線なのだから、
  原子データ自体の不確かさも同程度以上あると考えるのが自然。

## 6-4. 自分で G(T) を作る（fiasco、オプション）

**fiasco** は sunpy 系の CHIANTI インターフェース。
CHIANTI DB を持っているなら、そのまま G(T) を計算できる。

| | fiasco 0.8.2 | ChiantiPy 0.16.0 |
|---|---|---|
| CHIANTI IDL との一致 | ✅ そのまま 3% 以内 | ⚠ 4π の補正が要る |
| numpy 2.x | ✅ | ⚠ 単一温度で落ちる |
| バッチ実行 | ✅ | ⚠ `chiantirc` が無いと対話を要求（Colab で必ず踏む） |
| DB の入手 | `download_dbase()` で自動 | 手動 |

→ **講習会の本線は fiasco**。ChiantiPy は「IDL から来た人向けの補足」。

**DB のサイズ（実測）**: v9.0.1 = **257 MB**、v10.1 = 1058 MB、
v11.0.2（fiasco の既定）= 579 MB。
**v9.0.1 が最も軽く、しかも論文（CHIANTI 7）に最も近い。**

In [ ]:
USE_FIASCO = os.path.exists(os.path.expanduser("~/.fiasco/fiascorc"))
print("fiasco の設定がある" if USE_FIASCO else
      "fiasco の DB が無いので事前計算のファイルを使う（それで十分）")

if USE_FIASCO:
    import astropy.units as u
    import fiasco

    T = 10**logT * u.K
    ion = fiasco.Ion("Fe 12", T, abundance="sun_coronal_1992_feldman")
    cf = ion.contribution_function(1e9 * u.cm**-3)     # (nT, 1, n_transitions)
    k = int(np.argmin(np.abs(ion.transitions.wavelength[~ion.transitions.is_twophoton]
                            .to_value("Angstrom") - 195.119)))
    g_fiasco = cf[:, 0, k].to_value("erg cm3 / s") * 0.83   # n_H/n_e = 0.83
    g_ref = G[int(np.argmin(np.abs(wvl - 195.119)))]
    j = int(np.argmax(g_ref))
    print(f"\nFe XII 195.119 のピーク値")
    print(f"  fiasco       : {g_fiasco[int(np.argmax(g_fiasco))]:.3e}")
    print(f"  同梱ファイル : {g_ref[j]:.3e}   （CHIANTI IDL 由来）")
    print(f"  比           : {g_fiasco[int(np.argmax(g_fiasco))]/g_ref[j]:.3f}")
else:
    print("\n自分で作るなら:")
    print("  from fiasco.util import download_dbase")
    print("  download_dbase('http://download.chiantidatabase.org/"
          "CHIANTI_9.0.1_database.tar.gz', '/content/chianti')")
    print("  （257 MB。scripts/gofnt_fiasco.py が 22 輝線ぶんを作る）")

**22 輝線すべてで fiasco と CHIANTI IDL は median 1.000、最大 3.5% 差、
形成温度は完全一致**することを確認済み（`scripts/gofnt_fiasco.py`）。
→ **講習会は Python だけで寄与関数を出せる。**

## 6-5. ★ EM loci —— DEM を解く前に必ず描く図

輝線 λ の観測強度 $I_\lambda$ に対して、
「**もし視線上のプラズマが全部ちょうど温度 T にあったら**、必要な EM はいくらか」:

$$ {\rm EM}_{\rm loci,\lambda}(T) = \frac{4\pi I_\lambda}{G_\lambda(T)} $$

- これは各温度における **EM の上限**。真の EM は必ずこの曲線より**下**にある
- 全輝線の曲線を重ねると、**下側の包絡線**が DEM の目安になる
- **等温プラズマなら全曲線が 1 点で交わる。** 交わらなければ多温度

In [ ]:
# モジュール 2 が書いた強度を読む（無ければその場で作る）
import csv
import sys
sys.path.insert(0, "scripts")

# モジュール 2 の出力。無ければその場で作る（scripts/workshop.py、10 秒ほど）
from workshop import box_intensities

rows = box_intensities()

iobs = np.zeros(len(names))
for ion, w, i_fit, i_paper, ratio in rows:
    k = int(np.argmin(np.abs(wvl - w)))
    if abs(wvl[k] - w) < 0.01:
        iobs[k] = i_fit

# ★ Ca XVII 192.858 はブレンドしたままの値（論文の 5 倍）なので EM loci から外す。
#   モジュール 8 で分離したら戻す。
iobs[np.argmin(np.abs(wvl - 192.858))] = 0.0
print(f"EM loci に使う輝線: {int((iobs > 0).sum())} 本")

In [ ]:
logTf, namesf, wvlf, Gf = read_gofnt(GOFNT_FINE)      # 細かい格子の方が見やすい
iobsf = np.zeros(len(namesf))
for ion, w, i_fit, i_paper, ratio in rows:
    k = int(np.argmin(np.abs(wvlf - w)))
    if abs(wvlf[k] - w) < 0.01:
        iobsf[k] = i_fit
iobsf[np.argmin(np.abs(wvlf - 192.858))] = 0.0

fig, ax = plt.subplots(figsize=(8.5, 5.8))
cmap = plt.get_cmap("turbo")
ok = np.where(iobsf > 0)[0]
tpk = np.array([logTf[int(np.argmax(Gf[k]))] for k in ok])
norm = plt.Normalize(tpk.min(), tpk.max())

env = np.full(len(logTf), np.inf)
for k, tp in zip(ok, tpk):
    g = Gf[k]
    m = g > g.max() * 1e-3                     # G が十分ある温度だけ描く
    loci = 4 * np.pi * iobsf[k] / np.where(g > 0, g, np.nan)
    ax.plot(logTf[m], loci[m], color=cmap(norm(tp)), lw=1.3, alpha=0.9)
    j = int(np.nanargmin(np.where(m, loci, np.inf)))
    ax.annotate(f"{namesf[k]} {wvlf[k]:.1f}", (logTf[j], loci[j]), fontsize=6.5,
                color=cmap(norm(tp)), xytext=(2, 2), textcoords="offset points")
    env = np.minimum(env, np.where(m, loci, np.inf))

ax.plot(logTf, env, "k--", lw=1.8, label="lower envelope = upper limit on EM")
ax.set_yscale("log")
ax.set_xlim(5.4, 7.2)
ax.set_ylim(1e25, 1e31)
ax.set_xlabel("log T [K]")
ax.set_ylabel(r"EM$_{\rm loci} = 4\pi I / G(T)$   [cm$^{-5}$]")
ax.set_title("EM loci: EM required if ALL the plasma were at temperature T")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
             label="log T at peak of G(T)")
fig.tight_layout()
plt.show()

print(f"{'logT':>6} {'T [MK]':>8} {'包絡線 = EM の上限 [cm^-5]':>28}")
for t in (5.8, 6.0, 6.2, 6.4, 6.6, 6.8, 7.0):
    j = int(np.argmin(np.abs(logTf - t)))
    print(f"{t:6.1f} {10**t/1e6:8.2f} {env[j]:28.2e}")

**読み方**

1. **曲線は 1 点で交わらない** → **等温ではない**。多温度のプラズマがある。
   これが「DEM を解く」動機そのもの。
2. **包絡線は各温度における EM の上限**。真の EM 分布は必ずこの下にある。
3. **log T ≈ 5.8 の深い谷**（Si VII が作っている）。
   「もし全部 0.6 MK なら EM は 7.6e25 しか要らない」= **低温プラズマは非常に少ない**。
   inter-moss（ループ上部）を選んだ効果がここに出ている。
4. 包絡線は低温側から **6.8 まで単調に上がる**（7.6e25 → 7.4e27）。
   上限が緩いほど、その温度には EM があってよい。
   **多数の曲線の底が log T 6.1–6.6 に集まっている**のが本体の在りか。
   モジュール 7 で解く DEM のピーク（log T 6.6）は、
   そこでの上限 7.3×10²⁷ の下にちゃんと収まる。**これが検算になる。**
5. **log T = 7.0 で包絡線が 1.3×10²⁹ に跳ね上がる**。
   そこに効く輝線がもう無い = **上限が事実上つかない**ということ。
   → **7 MK 以上はこのデータだけでは決まらない**。だから AIA Fe XVIII を足す
     （入れないと DEM が高温側に漏れる。PINTofALE の文書が
      "toothpaste tube effect" と呼ぶ現象）。
6. EM のオーダーは **10²⁷–10²⁸ cm⁻⁵**。活動領域として妥当（6-3 の検算どおり）。

**★ この図を先に描いておくと、DEM の解が変になったときの検算に使える。**
逆問題の答えが包絡線を超えていたら、それだけで間違い。

## 6-6. 演習

1. `iobs` を全部 1.3 倍して EM loci を描き直す（較正が 30% 違ったら、の想定）。
   **形は変わるか、それとも上下に平行移動するだけか？**
   → 「較正誤差では説明できないパターン」の意味が体で分かる。
2. Ca XVII をブレンドしたままの値（論文の 5 倍）で EM loci に入れてみる。
   包絡線がどう壊れるか。**1 本の誤った線が高温側の結論をどう変えるか。**
3. G(T) のピーク温度が最も近い 2 本（Fe XI 180.401 と Fe XI 188.216）の
   EM loci 曲線を比べる。**同じイオンなら重なるはず**。重ならなければ何が違うのか。
4. 組成をコロナ組成から光球組成に変えたら G(T) はどう変わるか（発展、fiasco が要る）。
   Fe と Ca は低 FIP 元素、S と Ar は高 FIP 元素。

## まとめ

- G(T) = 組成 × 電離平衡 × 励起。**温度が狭いのは電離平衡のおかげ**
- 22 輝線が logT 5.8–6.9 に少しずつずれて効く。**でも幅広く重なっている**
- **1/(4π) の約束は実装ごとに違う**。エラーは出ない。オーダーで検算する
- **EM loci は逆問題を解く前に必ず描く**。答えの見当と検算に使える

次（モジュール 7）で、いよいよ **DEM の逆問題**を解く。

# モジュール 7: DEM インバージョン

**所要時間 60 分**

**このノートで身につくこと**

1. DEM 逆問題が **ill-posed** であることを、特異値分解で**数値として**見る
2. 誤差の床（較正の系統誤差）を入れる理由を理解する
3. `demregpy`（正則化）で実際に解き、論文 Table 2 の R 列と比べる
4. ★ **手法・設定で答えがどれだけ動くか**を測る（これがこの章の核心）
5. 傾き α から加熱の物理を議論する

前提: モジュール 2（強度）、4（AIA）、6（G(T)）。

In [ ]:
import csv
import os

import numpy as np
import matplotlib.pyplot as plt

GOFNT_FINE = "work/gofnt_chianti901_005.txt"   # 0.05 dex（demregpy 用）
GOFNT = "work/gofnt_chianti901.txt"            # 0.10 dex（MCMC 用）
AIARESP = "work/aia94_fe18_response.txt"
MCMC = "work/mcmc_dem_result.txt"              # PINTofALE MCMC_DEM の結果（同梱）


def read_gofnt(path):
    lines = open(path).readlines()
    i = next(k for k, l in enumerate(lines) if l.startswith("# nT nline"))
    nT, nline = (int(v) for v in lines[i + 1].split())
    k = i + 2

    def skip(tag):
        nonlocal k
        while not lines[k].startswith(tag):
            k += 1
        k += 1

    def take(n):
        nonlocal k
        out = []
        while len(out) < n:
            out += [float(x) for x in lines[k].split()]
            k += 1
        return np.array(out[:n])

    skip("# logT")
    logT = take(nT)
    skip("# ion")
    names, wvl = [], []
    for _ in range(nline):
        p = lines[k].split()
        names.append(" ".join(p[:-2]))
        wvl.append(float(p[-2]))
        k += 1
    skip("# G(T)")
    G = np.array([take(nT) for _ in range(nline)])
    return logT, names, np.array(wvl), G


logT, names, wvl, G = read_gofnt(GOFNT_FINE)
keep = (logT >= 5.5) & (logT <= 7.1)
logT, G = logT[keep], G[:, keep]
dlt = logT[1] - logT[0]
print(f"温度格子: logT {logT[0]:.2f}–{logT[-1]:.2f}, {len(logT)} ビン, {dlt:.2f} dex 刻み")

## 7-1. ★ なぜ ill-posed なのか —— 特異値で見る

解きたいのは

$$ I_\lambda = \frac{1}{4\pi}\int G_\lambda(T)\,\xi(T)\,dT $$

で、$\xi(T)$ が未知（**第一種 Fredholm 積分方程式**）。
温度を離散化すると、ただの行列方程式 $\mathbf{I} = \mathsf{A}\,\boldsymbol{\xi}$ になる。
$\mathsf{A}$ は「輝線 × 温度ビン」の行列で、中身は $G/(4\pi)$。

**この行列がどれくらい"効いて"いるかは特異値を見れば分かる。**

In [ ]:
A = G / (4 * np.pi)                 # (nline, nT)
sv = np.linalg.svd(A, compute_uv=False)
print(f"行列の形     : {A.shape}  (輝線 x 温度ビン)")
print(f"条件数       : {sv[0]/sv[-1]:.2e}")
print(f"特異値（上位）: {np.round(sv[:8]/sv[0], 4)}")
print(f"最大の 1/1000 以上ある特異値の本数 = {int((sv > 1e-3*sv[0]).sum())}")

plt.figure(figsize=(6, 4))
plt.semilogy(sv / sv[0], "o-")
plt.axhline(1e-3, color="r", ls="--", label="1/1000 of the largest")
plt.xlabel("index")
plt.ylabel("singular value / max")
plt.title("singular values of the response matrix")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**22 本の輝線を測ったのに、実質的な自由度は 12 程度しかない。**

理由はモジュール 6 で見たとおり、**G(T) が幅広く重なっている**から。
隣り合う温度ビンの応答がほとんど同じなので、両者を区別できない。

数学的には、特異値が急速に 0 に近づくと逆作用素が非有界になる。
実際問題としては:

- $\xi(T)$ に細かい構造を入れても、$G$ との積分で均されて $I_\lambda$ に現れない
- 逆に、$I_\lambda$ をわずかに動かすと $\xi$ の細かい構造は激しく変わる
  = **観測誤差が解に増幅される**

→ だから**何らかの追加の仮定（正則化・事前分布）が必ず要る**。
  「DEM を解いた」と言うときは、**何を仮定したか**を必ずセットで言う。

## 7-2. 観測強度と誤差

モジュール 2 で出した 22 輝線の強度を読む。

- **Ca XVII 192.858 は外す**（ブレンドしたまま。モジュール 8 で分離したら戻す）
- 誤差は **22%**（論文と同じ）。統計誤差は 0.2% しかないが、
  それを使うと χ² が発散する（モジュール 2 参照）

In [ ]:
# モジュール 2 の出力を読む。Colab はノート 1 冊ごとに VM が変わるので、
# 無ければその場で作り直す（`scripts/workshop.py` にまとめてある。10 秒ほど）。
from workshop import box_intensities

iobs = np.zeros(len(names))
for ion, w, i_fit, i_paper, ratio in box_intensities():
    k = int(np.argmin(np.abs(wvl - w)))
    if abs(wvl[k] - w) < 0.01:
        iobs[k] = i_fit
iobs[int(np.argmin(np.abs(wvl - 192.858)))] = 0.0        # Ca XVII を外す

ok = iobs > 0
labels = [f"{names[i]} {wvl[i]:.3f}" for i in np.where(ok)[0]]
dn = iobs[ok]
edn = 0.22 * dn                                          # ★ 較正の系統誤差を床に
tresp = (A[ok]).T                                        # (nT, nf)
print(f"EIS から {len(dn)} 本")

### AIA Fe XVIII を拘束に加える

モジュール 6 で見たとおり、**log T > 6.9 は EIS だけでは決まらない**。
入れないと DEM が高温側に漏れて傾き β が出鱈目になる
（PINTofALE の文書が **"toothpaste tube effect"** と呼ぶ現象）。

**★ 公式の AIA 94 Å 応答は使えない。** 低温線の寄与が入っているため。
**Fe XVIII だけの応答関数**を作ってある
（`scripts/aia94_fe18_response.py`。CHIANTI の Fe XVIII 582 本を
aiapy の波長応答に畳み込んだもの。実際は 93.932 Å の 1 本が寄与の 100%）。

In [ ]:
d = np.loadtxt(AIARESP)
R = np.interp(logT, d[:, 0], d[:, 1])
print(f"R(T) のピーク = {R.max():.3e} DN cm^5 s^-1 pix^-1 "
      f"at logT {logT[int(np.argmax(R))]:.2f}")

# モジュール 4 の出力。無ければその場で作り直す（AIA 3 MB の取得込みで十数秒）
from workshop import aia_on_eis_grid, BOX

npz = aia_on_eis_grid()
aia = float(np.nanmean(npz["fe18"][BOX["y0"]:BOX["y1"], BOX["x0"]:BOX["x1"]]))
print(f"AIA Fe XVIII（同じ箱の平均） = {aia:.2f} DN/s  （論文 Table 2 = 7.20）")

tresp = np.column_stack([tresp, R])
dn = np.append(dn, aia)
edn = np.append(edn, 0.19 * aia)                          # 論文の 1.40/7.20 = 19%
labels.append("AIA 94 FeXVIII")
print(f"拘束の合計 = {len(dn)} 本")

**★ 単位のつじつま（3 回踏んだので明示する）**

| | 単位 | demregpy に渡すもの |
|---|---|---|
| EIS 輝線 | $I$ = erg cm⁻² s⁻¹ sr⁻¹、$G$ = erg cm³ s⁻¹ | **$G/(4\pi)$** |
| AIA | DN s⁻¹ pix⁻¹ | **$R(T)$ そのまま**（1/4π と画素立体角込み） |

返ってくる DEM は **[cm⁻⁵ K⁻¹]**。

## 7-3. 解く

In [ ]:
from demregpy import dn2dem

T = 10**logT
tedges = 10 ** np.append(logT - dlt / 2, logT[-1] + dlt / 2)


def solve(**kw):
    """解いて (ビンあたりの EM, chi2_red, モデル強度) を返す。"""
    de, _, _, ch, dr = dn2dem(dn, edn, tresp, logT, tedges, max_iter=30,
                              warn=False, **kw)
    de = np.atleast_1d(np.squeeze(de))
    # ★ demregpy の DEM は [cm^-5/K]。ビンあたりの EM にするには ΔT を掛ける
    return (de * T * np.log(10) * dlt, float(np.squeeze(ch)),
            np.atleast_1d(np.squeeze(dr)))


em, chi2, dn_reg = solve()
print(f"reduced chi2 = {chi2:.2f}")
print(f"EM ピーク    = logT {logT[int(np.nanargmax(em))]:.2f} "
      f"({T[int(np.nanargmax(em))]/1e6:.2f} MK)")
print(f"総 EM        = {em.sum():.2e} cm^-5")
print(f"  → n_e = 1e9 cm^-3 とすると視線長 L = {em.sum()/1e18/1e8:.0f} Mm")
print("  （EM = n_e^2 L。活動領域の視線長として妥当なオーダー）")

## 7-4. R = I_obs / I_DEM のパターンを論文と比べる

論文 Table 2 の R 列がこれ。**個々の線がどれだけ再現できたか**を見る。

In [ ]:
# 比較のため、事前分布に MCMC の解を入れた解（χ² が最も小さい）も出す
m0 = np.loadtxt(MCMC)
norm_mcmc = np.interp(logT, m0[:, 0], m0[:, 2])
norm_mcmc = norm_mcmc / norm_mcmc.max()
em_prior, chi2_prior, dn_prior = solve(dem_norm0=norm_mcmc)
print(f"既定の解 chi2 = {chi2:.2f}   MCMC を事前分布にした解 chi2 = {chi2_prior:.2f}\n")

# 論文 Table 2 の R 列（= I_obs / I_dem）をそのまま転記したもの
PAPER_R = {"Si VII 275.368": 1.10, "Fe IX 188.497": 1.01, "Fe IX 197.862": 0.93,
           "Fe X 184.536": 1.40, "Fe XI 180.401": 0.88, "Fe XI 188.216": 1.11,
           "S X 264.233": 1.00, "Si X 258.375": 0.78, "Fe XII 192.394": 1.01,
           "Fe XII 195.119": 1.04, "Fe XIII 202.044": 1.80, "Fe XIII 203.826": 1.82,
           "Fe XIV 264.787": 0.90, "Fe XIV 270.519": 0.90, "Fe XV 284.160": 0.81,
           "S XIII 256.686": 0.91, "Fe XVI 262.984": 1.04, "Ar XIV 194.396": 1.36,
           "Ca XIV 193.874": 1.31, "Ca XV 200.972": 1.43, "Ca XVI 208.604": 0.75,
           "AIA 94 FeXVIII": 0.98}
print(f"{'line':<20}{'I_obs':>10}{'R (既定)':>10}{'R (MCMC事前)':>13}{'論文 R':>8}")
for lab, o, p1, p2 in zip(labels, dn, dn_reg, dn_prior):
    pr = PAPER_R.get(lab)
    print(f"{lab:<20}{o:10.2f}{o/p1:10.2f}{o/p2:13.2f}"
          f"{(f'{pr:.2f}' if pr else ''):>8}")

**★ まず、R そのものが事前分布で動くことに注目。**
既定の解では R が全体に 1.2–4 と大きいが、
χ² の小さい（＝よく合っている）解では 1.0–2 に収まる。
**「どの線が合わないか」自体が、解き方に依存する。**

それでも**パターンは共通**していて、しかも論文と同じ形をしている:

1. **Fe XIII の 203.826 が突出して外れる**（3–4）。
   論文でも 1.80 / 1.82、Warren et al. (2011) でも 1.87 / 1.90 と外れている。
   → **原子データ側の既知の問題**。モジュール 6 で見たとおり、
     Fe XIII は**実装間の差が最大の線**（密度敏感線）でもある。
     「コードが違うだけで 3–12% 変わる線」なのだから、
     原子データ自体の不確かさも同程度以上あると考えるのが自然。
2. **Ar XIV / Ca XIV / Ca XV が揃って上がる**（1.6–2.1）。
   論文も 1.36 / 1.31 / 1.43 と**揃って**上がっている。
   → 高温側の輝線が系統的に「DEM で説明しきれない」。
     組成（Ar は高 FIP、Ca は低 FIP）や電離平衡が疑われる。
3. Fe XIII を除く 1–2 MK の鉄の線（Fe XI, XII, XIV, XV）は
   0.8–1.2 に収まる。ここは論文とほぼ同じ水準。

**合わない線も見ておく。** 我々は Fe IX 197.862（1.47）と
Fe X 184.536（1.45）が高いが、論文は 0.93 と 1.40 で片方しか外れていない。
低温側は箱の選び方（moss をどれだけ含むか）に最も敏感な部分で、
**モジュール 5 で見た Si VII の問題と地続き**である。

**★ ここが重要**: 我々の R のパターンが論文と**同じ形**をしている。
絶対強度が 11% 低くても、**DEM 解析の結論は論文と同じ**になる。

（なお PINTofALE の MCMC_DEM で解くと、
 Ar XIV / Ca XIV / Ca XV が **1.37 / 1.35 / 1.46** と
 論文の 1.36 / 1.31 / 1.43 をほぼそのまま再現する。
 正則化とは別の手法で、より論文に近い R が出る。）

## 7-5. ★★ 手法・設定で答えがどれだけ動くか

ここがこの章の核心。**同じ観測強度・同じ G(T)** で設定だけを変える。

- `reg_tweak`: 目標 χ²（大きいほど強く平滑化）
- `dem_norm0`: 初期の重み（＝事前分布）。**MCMC の解を入れてみる**

比較相手として、論文と同じ **PINTofALE の MCMC_DEM**
（Kashyap & Drake 1998）の結果を同梱してある（`work/mcmc_dem_result.txt`）。

In [ ]:
def slope(em, t0, t1):
    m = (logT >= t0) & (logT <= t1) & (em > 0)
    return np.polyfit(logT[m], np.log10(em[m]), 1)[0]


# (日本語の説明, 図の凡例（英語）, demregpy に渡す設定)
runs = [
    ("demregpy 既定",              "demregpy default",        dict()),
    ("demregpy reg_tweak=2",       "demregpy reg_tweak=2",    dict(reg_tweak=2.0)),
    ("demregpy gloci=1",           "demregpy gloci=1",        dict(gloci=1)),
    ("demregpy (MCMC を事前分布に)", "demregpy + MCMC prior",   dict(dem_norm0=norm_mcmc)),
]
results = {}
print(f"{'設定':<30}{'chi2':>7}{'EM ピーク':>12}{'alpha':>8}{'beta':>8}")
for tag, en, kw in runs:
    em_, ch, _ = solve(**kw)
    results[en] = em_
    ip = int(np.nanargmax(em_))
    print(f"{tag:<30}{ch:7.2f}{logT[ip]:8.2f} ({T[ip]/1e6:.1f}MK)"
          f"{slope(em_, 6.0, 6.6):+8.2f}{-slope(em_, 6.6, 7.0):+8.2f}")

# MCMC（PINTofALE）: DEM は [cm^-5/logK] なので EM = DEM × ΔlogT
dlt_m = float(np.median(np.diff(m0[:, 0])))
lt_m, em_m = m0[:, 0], m0[:, 2] * dlt_m
ipm = int(np.nanargmax(em_m))


def slope_m(t0, t1):
    m = (lt_m >= t0) & (lt_m <= t1) & (em_m > 0)
    return np.polyfit(lt_m[m], np.log10(em_m[m]), 1)[0]


print(f"{'MCMC_DEM (PINTofALE)':<30}{'—':>7}{lt_m[ipm]:8.2f} "
      f"({10**lt_m[ipm]/1e6:.1f}MK){slope_m(6.0, 6.6):+8.2f}{-slope_m(6.6, 7.0):+8.2f}")
print(f"{'論文 Table 1 region 7':<30}{'—':>7}{'~6.6 (4 MK)':>12}{2.9:+8.2f}{9.0:+8.2f}")

In [ ]:
plt.figure(figsize=(8, 5.5))
for tag, em_ in results.items():
    plt.plot(logT, em_, "-", lw=1.6, label=tag)
plt.plot(lt_m, em_m, "k-", lw=2.6, marker="o", ms=4, label="MCMC_DEM (PINTofALE)")
plt.yscale("log")
plt.xlim(5.6, 7.1)
plt.ylim(1e24, 1e28)
plt.xlabel("log T [K]")
plt.ylabel(r"EM per bin [cm$^{-5}$]")
plt.title("same data, same G(T) — only the method/prior differs")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7-6. 読み取れること

1. **EM ピークの 4 MK 付近は、どの手法・どの設定でも動かない。**
   論文の主張の核（活動領域コアの EM は 4 MK に強くピークを持つ）は
   **手法に依らず頑健**。
2. **傾き α は設定で 1.5 – 2.3 と大きく動く。**
   正則化は既定の重みだとピークを均して α を過小評価する。
3. **MCMC の解を事前分布 `dem_norm0` に与えると χ² が改善し、α も MCMC に近づく。**
   → **答えが事前分布に強く依存している**証拠。
4. **どの設定でも χ² が 1 に届かない。**
   22 本・誤差 22% のデータを、滑らかな DEM で説明しきれていない。

論文自身もこう書いている:

> It is also clear, however, that the detailed structure of the EM distributions
> is much more difficult to determine with confidence.

**これを実データで再現したのがこの章の成果。**

## 7-7. ★ 単位の罠: per K か per logK か

**コードによって DEM の単位が違う。**

| | DEM の単位 | ビンあたりの EM |
|---|---|---|
| PINTofALE (MCMC) | cm⁻⁵ **/ logK** | DEM × ΔlogT |
| demregpy | cm⁻⁵ **/ K** | DEM × ΔT = DEM × T ln10 ΔlogT |

揃えないと **10⁷ ずれる**（実際に踏んだ）。

さらに**傾きにも効く**。論文 Eq.(3) の EM 分布 ξ(Te)dTe は
ξ が [cm⁻⁵/K] なので

$$ \xi\,dT_e = \frac{\rm DEM_{per\,logK}}{T\ln 10}\times(T\ln 10\ d\log T)
   = {\rm DEM_{per\,logK}}\times d\log T $$

**T は掛からない。** 準備段階でここを間違えて `DEM × T` を EM 分布として
傾きを測り、**α を 3.30 と誤って報告した（正しくは 2.30）**。
T を余計に掛けると傾きがちょうど **+1 ずれる**。

In [ ]:
em_correct = results["demregpy + MCMC prior"]
em_wrong = em_correct * T                     # わざと間違える
print(f"正しい   alpha = {slope(em_correct, 6.0, 6.6):+.2f}")
print(f"T を余計に掛ける = {slope(em_wrong, 6.0, 6.6):+.2f}   ← ちょうど +1 ずれる")
print("→ ピーク温度は動かないので、傾きだけ見ていると気づけない")

## 7-8. 傾き α は何を語るか —— 加熱の物理

ここが論文の科学的な主張。

**Parker のナノフレア説**: 磁力線が対流でランダムに揺すられ、ねじれが溜まり、
磁気リコネクションで**間欠的に**エネルギーを放出する。

ループの冷却時間 τ_cool と加熱イベントの間隔 τ_heat を比べると:

| | 描像 | EM 分布 |
|---|---|---|
| **低頻度加熱** (τ_heat ≫ τ_cool) | ナノフレア。加熱後に十分冷える | 幅広い。**α は緩い**（≲ 2.3） |
| **高頻度加熱** (τ_heat ≪ τ_cool) | ほぼ定常。冷える暇がない | 鋭くピーク。**α は急** |

論文が観測した α ≈ 2.9–3.4 は、
ナノフレア・シミュレーションの上限 2.3 より**急**。
→ 「活動領域コアの高温プラズマは熱平衡に近い」
→ 「加熱は**高頻度**でなければならない」
→ Parker のナノフレア描像（低頻度）への挑戦。

### ★ 我々の結果が持つ意味（正直に）

我々の α は **1.5–2.3**。MCMC で 2.30、これは
**ナノフレア・シミュレーションの上限とちょうど同じ**。
論文の 2.9 は明確に上回るが、我々の値は**境界上にある**。

つまり **α の 0.6 の差が科学的な結論を左右する**。そして α は

- 箱の位置（median が 0.83–0.95 で動く範囲でも変わる）
- DEM の手法・正則化の設定（**1.5–2.3**）
- 温度ビンの取り方

に敏感。**「α が急だからナノフレアは否定される」と言うには、
これらの系統誤差を全部押さえる必要がある。**
これは論文への批判ではなく、**この種の測定の難しさそのもの**。
講習会で一番伝えたいのはここ。

## 7-9. 実装上の注意（踏むと分からない）

1. **温度ビンの粗さの要求が、2 手法で正反対。**

   | | MCMC_DEM | demregpy（正則化） |
   |---|---|---|
   | 温度ビン | **粗い方が良い**（0.1 dex） | **細かい方が良い**（0.05 dex） |
   | 理由 | 各ビンが独立パラメータ。劣決定だと解が跳ねる | 平滑化項が劣決定を吸収する。**ビン数 > 拘束数が必須** |

   demregpy に 17 ビン・23 拘束を渡すと GSVD が
   `ValueError: operands could not be broadcast together` で破綻する。
   **正則化は劣決定を前提にする手法**だという性格がそのまま出ている。

2. **MCMC の `dem` は χ² 最小の 1 実現**なので凸凹する。
   論文が描いているのは MCMC アンサンブルなので、
   `simdem` から中央値を取る（同梱ファイルはそうしてある）。

3. **誤差に較正の床を入れないと χ² が発散する**（モジュール 2）。

## 7-10. 演習

1. **AIA Fe XVIII を外して**解き直す。高温側の傾き β がどうなるか。
   （"toothpaste tube effect" を自分で見る）
2. 誤差を 22% → 10% にすると χ² と解はどう変わるか。
   **誤差を小さく見積もることの危険**を体感する。
3. Fe XIII の 2 本を拘束から外して解き直す。
   他の線の R は改善するか？ **1 つの外れ値が全体をどれだけ引っ張るか。**
4. 温度範囲を `logT <= 6.8` に狭めると何が起きるか。
5. **EM loci（モジュール 6）の包絡線の下に、解いた EM が収まっているか**確認する。
   超えていたらそれだけで間違い。

## まとめ

- DEM 逆問題は **ill-posed**。22 本測っても実質的な自由度は 12 程度
- だから**必ず追加の仮定が要る**。「解いた」と言うときは仮定もセットで言う
- **EM ピーク 4 MK は手法に依らず頑健**。**傾き α は手法で 1.5–2.3 と動く**
- R のパターン（Fe XIII が外れ、Ar XIV/Ca XIV/Ca XV が揃って上がる）は
  **論文と同じ形**。絶対強度が 11% 低くても結論は変わらない
- **単位（per K / per logK）を間違えると傾きが +1 ずれる**

以上で「1 日コース」が完走。
発展（モジュール 8–10）では Ca XVII のブレンド分離、較正の効き、
他の活動領域へ進む。